# Thesis code for my bioinformatic workflow. My lab notebook usually links to the orginal files of code but this is the full list of all code.

After copying and pasting the modomics table in a sheet and inputting it into this code. This code will then parse uniprot for each pfam ID and records all accession IDs and taxanomy info. Took about 50 hours to run.

In [ ]:
import pandas as pd
from Bio import ExPASy
from Bio import SwissProt
import requests
import time
import pickle
import os


INPUT_EXCEL_FILE = "RNA Modification in Viruses.xlsx"
PFAM_ID_COLUMN_NAME = "Pfam ID"
OUTPUT_EXCEL_FILE = "Pfam_UniProt_Taxonomy_Data.xlsx"
GOOGLE_DRIVE_MOUNT_PATH = "/content/drive/MyDrive"
PICKLE_DIR = os.path.join(GOOGLE_DRIVE_MOUNT_PATH, "Pfam Pickles")

# Base URL for UniProt search
UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

# Methylase skipping
METHYLASE_PFAM_IDS_TO_SKIP = []
METHYLASE_KEYWORDS = ["methyltransferase", "methylase", "m6a", "m1g", "m5c", "rrmj", "trm"]

#uploading pfam IDs from excel
def load_pfam_ids(excel_file, column_name):
    print(f"Reading PFAM IDs from '{excel_file}'...")
    try:
        df_input = pd.read_excel(excel_file)
    except FileNotFoundError:
        print(f"Error: Input file '{excel_file}' not found. Please ensure it's in the correct directory.")
        return []
    except pd.errors.EmptyDataError:
        print(f"Error: '{excel_file}' is empty.")
        return []

    if column_name not in df_input.columns:
        raise ValueError(f"Column '{column_name}' not found in '{excel_file}'. "
                         "Please check the column name in your Excel file.")

    pfam_ids = df_input[column_name].dropna().unique().tolist()
    print(f"Found {len(pfam_ids)} unique PFAM IDs to process.")
    return [str(pid).strip() for pid in pfam_ids] # Ensure IDs are stripped strings for consistency

#pickle maker, to save all pickles to drive folder named Pfam Pickles
def save_to_pickle(data, pfam_id, directory):
    try:
        os.makedirs(directory, exist_ok=True)
        pickle_file_path = os.path.join(directory, f"pfam_{pfam_id}.pkl")
        with open(pickle_file_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"  Data for {pfam_id} saved to {pickle_file_path}")
    except Exception as e:
        print(f"  Error saving pickle file for {pfam_id}: {e}")

#function to load pickles and check if there is already saved data, if not then fetch new ones
def load_from_pickle(pfam_id, directory):
    pickle_file_path = os.path.join(directory, f"pfam_{pfam_id}.pkl")

    # check if the file path exists and is a non empty file
    if os.path.exists(pickle_file_path) and os.path.getsize(pickle_file_path) > 0:
        print(f"  Cache found for {pfam_id}. Attempting to load from {pickle_file_path}...")
        try:
            with open(pickle_file_path, 'rb') as f:
                return pickle.load(f)
        except (EOFError, pickle.UnpicklingError) as e:
            # This handles cases where the file is empty or corrupted.
            print(f"  Error loading from corrupted/empty pickle file for {pfam_id} ({e}). Will re-fetch data.")
            return None
        except Exception as e:
            # other potential loading errors
            print(f"  Unexpected error loading pickle for {pfam_id}: {e}. Will re-fetch data.")
            return None
    return None

#gets all uniprot accessions for a pfam ID
def fetch_uniprot_search_results(pfam_id, base_url, fields, size=500):
    query = f"xref:PFAM-{pfam_id}"
    initial_url = f"{base_url}?query={query}&format=json&fields={fields}&size={size}"
    all_results = []
    current_url = initial_url
    while current_url:
        print(f"  Fetching UniProt search page...")
        try:
            response = requests.get(current_url)
            response.raise_for_status()
            data = response.json()

            all_results.extend(data.get('results', []))
            # Find the URL for the next page of results
            next_link = None
            for link in data.get('links', []):
                if link.get('rel') == 'next':
                    next_link = link.get('url')
                    break
            current_url = next_link
            if current_url:
                time.sleep(1) # Add a small delay between requests

        except requests.exceptions.RequestException as e:
            print(f"  Error fetching page from UniProt for PFAM {pfam_id}: {e}")
            break # Stop fetching if there's a request error

    return all_results

# checks both'recommendedName' and 'submissionNames' for proteins
def extract_protein_name_from_uniprot_result(result):
    protein_name_raw = ""
    # Try to get the recommended name first
    recommended_name_parts = result.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', [])
    if isinstance(recommended_name_parts, list):
        for item in recommended_name_parts:
            if isinstance(item, dict) and 'value' in item:
                protein_name_raw = item['value'].lower()
                break
    # Fallback to submission names if recommended name is not found
    if not protein_name_raw:
        submission_names = result.get('proteinDescription', {}).get('submissionNames', [])
        if isinstance(submission_names, list):
            for item in submission_names:
                if isinstance(item, dict) and 'fullName' in item:
                    full_name_parts = item.get('fullName', [])
                    if isinstance(full_name_parts, list):
                        for sub_item in full_name_parts:
                            if isinstance(sub_item, dict) and 'value' in sub_item:
                                protein_name_raw = sub_item['value'].lower()
                                break
                        if protein_name_raw:
                            break
    return protein_name_raw

#Fetches a full Swiss-Prot record using ExPASy
def get_swissprot_record_with_retry(uniprot_accession, max_retries=3):
    record = None
    for attempt in range(max_retries):
        try:
            with ExPASy.get_sprot_raw(uniprot_accession) as handle:
                record = SwissProt.read(handle)
            break # Break the loop if successful
        except Exception as e:
            print(f"    Error fetching SwissProt for {uniprot_accession} (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2) # Wait before retrying
            else:
                record = None # Set to None if all retries fail
    return record

#fetches NCBI taxanomy ID from swissprot
def extract_taxonomy_id(record):
    taxonomy_id = None
    if record and record.taxonomy_id:
        taxonomy_id = ", ".join(record.taxonomy_id)
    else:
        # Fallback to looking in cross-references if the direct attribute is empty
        for xref in record.cross_references:
            if len(xref) >= 3 and 'taxon:' in xref[2]:
                try:
                    taxon_str_parts = [part.strip() for part in xref[2].split(';') if 'taxon:' in part]
                    if taxon_str_parts:
                        taxonomy_id = taxon_str_parts[0].split(':')[-1].strip()
                        break
                except Exception:
                    # Continue if parsing a specific cross-reference fails
                    pass
    return taxonomy_id

# Processes a single UniProt accession to get detailed Swiss-Prot data.
def process_uniprot_accession(pfam_id, uniprot_accession, protein_name_from_search):
    # Skip based on keywords found in the initial search result
    if any(keyword in protein_name_from_search for keyword in METHYLASE_KEYWORDS):
        print(f"  Skipping UniProt Accession: {uniprot_accession} (identified as methylase by keyword)")
        return None

    print(f"  Fetching detailed data for UniProt Accession: {uniprot_accession}")
    record = get_swissprot_record_with_retry(uniprot_accession)

    if record:
        taxonomy_id = extract_taxonomy_id(record)
        return {
            'PFAM ID': pfam_id,
            'UniProt Accession': uniprot_accession,
            'Entry Name': record.entry_name,
            'Protein Name': record.description,
            'Organism': record.organism,
            'Taxonomy ID': taxonomy_id,
            'Sequence Length': record.sequence_length,
            'Taxonomy': record.organism_classification,
            'Superkingdom': record.organism_classification[0] if record.organism_classification else None,
        }
    else:
        print(f"    Could not retrieve full record for {uniprot_accession}. Appending a placeholder.")
        return {
            'PFAM ID': pfam_id,
            'UniProt Accession': uniprot_accession,
            'Entry Name': None, 'Protein Name': None, 'Organism': None,
            'Taxonomy ID': None, 'Sequence Length': None, 'Taxonomy': None,
            'Superkingdom': None
        }

# Main function to fetch or load data for a single PFAM ID.
def get_uniprot_data_for_pfam(pfam_id):
    print(f"\n--- Processing PFAM ID: {pfam_id} ---")

    # Check if the data for this PFAM ID is already cached.
    cached_data = load_from_pickle(pfam_id, PICKLE_DIR)
    if cached_data is not None:
        # If cached data exists, we don't need to re-fetch anything.
        return cached_data

    #If no cached data exists, fetch from UniProt.
    print(f"  No valid cache found for {pfam_id}. Fetching from UniProt.")
    all_accessions_data_for_pfam = []
    try:
        # Fetch initial search results for all accessions linked to this PFAM ID
        search_results = fetch_uniprot_search_results(pfam_id, UNIPROT_BASE_URL, "accession,id,protein_name")

        # Process each accession found in the search results
        for result in search_results:
            uniprot_accession = result['primaryAccession']
            protein_name_from_search = extract_protein_name_from_uniprot_result(result)

            # Check for skipping based on explicit PFAM ID or protein name keywords
            if str(pfam_id) in METHYLASE_PFAM_IDS_TO_SKIP:
                print(f"  Skipping all UniProt accessions for PFAM ID {pfam_id} (explicitly skipped PFAM).")
                continue

            processed_data = process_uniprot_accession(pfam_id, uniprot_accession, protein_name_from_search)
            if processed_data:
                all_accessions_data_for_pfam.append(processed_data)

    except Exception as e:
        print(f"An unexpected error occurred while fetching data for PFAM {pfam_id}: {e}")

    # Save the newly fetched data to a pickle file for future runs.
    save_to_pickle(all_accessions_data_for_pfam, pfam_id, PICKLE_DIR)
    return all_accessions_data_for_pfam


def main():
    """
    Main function to orchestrate the entire data fetching, processing, and saving pipeline.
    """
    # Make sure the Google Drive directory is accessible.
    print(f"Checking for pickle directory at: {PICKLE_DIR}")
    if not os.path.exists(PICKLE_DIR):
        print(f"Pickle directory does not exist. Creating it: {PICKLE_DIR}")
        os.makedirs(PICKLE_DIR, exist_ok=True)
        time.sleep(2)
    else:
        print("Pickle directory found.")

    try:
        # Load the list of PFAM IDs from the Excel file.
        pfam_ids = load_pfam_ids(INPUT_EXCEL_FILE, PFAM_ID_COLUMN_NAME)

        all_results_list = []
        # Iterate through each PFAM ID
        for pfam_id in pfam_ids:
            results_for_id = get_uniprot_data_for_pfam(pfam_id)
            all_results_list.extend(results_for_id)
            time.sleep(1)

        # Consolidate all collected data into a single DataFrame.
        if all_results_list:
            final_df = pd.DataFrame(all_results_list)
            desired_columns = [
                'PFAM ID', 'UniProt Accession', 'Entry Name', 'Protein Name', 'Organism',
                'Taxonomy ID', 'Sequence Length', 'Taxonomy', 'Superkingdom'
            ]
            final_df = final_df[desired_columns]

            #Save the final DataFrame to an Excel file.
            final_df.to_excel(OUTPUT_EXCEL_FILE, index=False)
            print(f"\nData successfully saved to '{OUTPUT_EXCEL_FILE}'")
        else:
            print("\nNo data was collected. The output Excel file will not be created.")

    except ValueError as ve:
        print(f"Configuration Error: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

#entrypoint
if __name__ == "__main__":
    main()

Checking for pickle directory at: /content/drive/MyDrive/Pfam Pickles
Pickle directory found.
Reading PFAM IDs from 'RNA Modification in Viruses.xlsx'...
Found 177 unique PFAM IDs to process.

--- Processing PFAM ID: PF03481 ---
  Cache found for PF03481. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF03481.pkl...

--- Processing PFAM ID: PF13649 ---
  Cache found for PF13649. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF13649.pkl...

--- Processing PFAM ID: PF05063 ---
  Cache found for PF05063. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF05063.pkl...

--- Processing PFAM ID: PF09445 ---
  Cache found for PF09445. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF09445.pkl...

--- Processing PFAM ID: PF01134 ---
  Cache found for PF01134. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF01134.pkl...

--- Processing PFAM ID: PF03919 ---
  Cache found for PF03919. Attempting to load from 

this code links the pfam id and all superkingdom counts to each original name and ID and spits out an excel sheet with all the info

In [ ]:
import pandas as pd
import os


INPUT_VIRUS_FILE = "RNA Modification in Viruses.xlsx"
INPUT_SUMMARY_FILE = "Superkingdom_Counts_Summary.xlsx"
OUTPUT_COMBINED_FILE = "full names and superkingdom counts.xlsx"

def combine_data_and_save(virus_file, summary_file, output_file):
    """
    Combines data from the virus modification Excel file and the
    superkingdom counts summary, then saves the combined data to a new Excel file.
    """
    print(f"Attempting to load data from '{virus_file}' and '{summary_file}'...")


    try:
        df_virus = pd.read_excel(virus_file)
        df_virus_selected = df_virus[['ID', 'Traditional Name', 'Pfam ID', 'Full name']].copy()
        df_virus_selected['Pfam ID'] = df_virus_selected['Pfam ID'].astype(str).str.strip()
        print(f"Successfully loaded '{virus_file}'.")
    except FileNotFoundError:
        print(f"Error: Input file '{virus_file}' not found. Please ensure it is uploaded directly to your Colab session.")
        return
    except KeyError as e:
        print(f"Error: Missing expected column in '{virus_file}': {e}. Please check column names.")
        return
    except Exception as e:
        print(f"An unexpected error occurred while loading '{virus_file}': {e}")
        return


    try:
        df_summary = pd.read_excel(summary_file)

        df_summary['Pfam ID'] = df_summary['Pfam ID'].astype(str).str.strip()
        print(f"Successfully loaded '{summary_file}'.")
    except FileNotFoundError:
        print(f"Error: Input file '{summary_file}' not found. Please ensure it is uploaded directly to your Colab session.")
        return
    except Exception as e:
        print(f"An unexpected error occurred while loading '{summary_file}': {e}")
        return


    print("Merging dataframes based on 'Pfam ID'...")
    combined_df = pd.merge(df_virus_selected, df_summary, on='Pfam ID', how='left')

    combined_df['Superkingdom'] = combined_df['Superkingdom'].fillna('N/A')
    combined_df['Count'] = combined_df['Count'].fillna(0).astype(int)


    final_column_order = [
        'ID', 'Traditional Name', 'Pfam ID', 'Full name',
        'Superkingdom', 'Count'
    ]
    combined_df = combined_df[final_column_order]

    try:
        combined_df.to_excel(output_file, index=False)
        print(f"\nSuccessfully created and saved the combined data to '{output_file}'.")
    except Exception as e:
        print(f"Error saving the combined data to '{output_file}': {e}")


if __name__ == "__main__":

    combine_data_and_save(INPUT_VIRUS_FILE, INPUT_SUMMARY_FILE, OUTPUT_COMBINED_FILE)

Attempting to load data from 'RNA Modification in Viruses.xlsx' and 'Superkingdom_Counts_Summary.xlsx'...
Successfully loaded 'RNA Modification in Viruses.xlsx'.
Successfully loaded 'Superkingdom_Counts_Summary.xlsx'.
Merging dataframes based on 'Pfam ID'...

Successfully created and saved the combined data to 'full names and superkingdom counts.xlsx'.


this will code for pickles that store data about the rna modifications including substrates and products of each then output an excel file with everything. This excel file puts both substrates and products in one column so I had to go in by hand and correct this.

In [ ]:
import pandas as pd
import os
import pickle
import requests
import time
import re
from Bio import ExPASy
from Bio import SwissProt

INPUT_EXCEL_FILE = "RNA Modification in Viruses.xlsx"
ENZYME_NAME_COLUMN = "UniProt"
OUTPUT_EXCEL_FILE = "Enzyme_Catalytic_Activity.xlsx"

# Google Drive path and local directory for saving and loading pickle files
GOOGLE_DRIVE_MOUNT_PATH = "/content/drive/MyDrive"
PICKLE_DIR = os.path.join(GOOGLE_DRIVE_MOUNT_PATH, "Catalytic Pickles")

# Base URL for UniProt search
UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

#test functions to make sure the excel file can be made
def create_mock_excel(filename=INPUT_EXCEL_FILE):
    print(f"Creating mock Excel file: '{filename}'...")
    data = {
        ENZYME_NAME_COLUMN: [
            "tRNA (uracil-5-)-methyltransferase",
            "Ribosomal RNA (guanine-N1)-methyltransferase",
            "RNA (adenosine-N6)-methyltransferase",
            "DNA (adenine-N6)-methyltransferase",
            "nsp13",  # An example that might have a different search result format
            "Trm82"   # An example that caused a previous error, now handled
        ]
    }
    df = pd.DataFrame(data)
    df.to_excel(filename, index=False)
    print("Mock Excel file created successfully.")

#check if there is already a pickle for the enzyme
def load_enzyme_names(excel_file, column_name):

    print(f"Reading enzyme names from '{excel_file}'...")
    try:
        df_input = pd.read_excel(excel_file)
    except FileNotFoundError:
        print(f"Error: Input file '{excel_file}' not found. Please ensure it's in the correct directory.")
        return []
    except pd.errors.EmptyDataError:
        print(f"Error: '{excel_file}' is empty.")
        return []

    if column_name not in df_input.columns:
        raise ValueError(f"Column '{column_name}' not found in '{excel_file}'. "
                         "Please check the column name in your Excel file.")

    enzyme_names = df_input[column_name].dropna().unique().tolist()
    print(f"Found {len(enzyme_names)} unique enzymes to process.")
    return [str(name).strip() for name in enzyme_names]

#saving new pickle to folder
def save_to_pickle(data, enzyme_name, directory):

    try:
        os.makedirs(directory, exist_ok=True)
        filename = "".join(c for c in enzyme_name if c.isalnum() or c in (' ', '_', '-')).rstrip()
        filename = filename.replace(' ', '_')
        pickle_file_path = os.path.join(directory, f"{filename}.pkl")

        with open(pickle_file_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"  Data for '{enzyme_name}' saved to {pickle_file_path}")
    except Exception as e:
        print(f"  Error saving pickle file for '{enzyme_name}': {e}")

#if there is already a pickle it will load it
def load_from_pickle(enzyme_name, directory):

    filename = "".join(c for c in enzyme_name if c.isalnum() or c in (' ', '_', '-')).rstrip()
    filename = filename.replace(' ', '_')
    pickle_file_path = os.path.join(directory, f"{filename}.pkl")

    if os.path.exists(pickle_file_path) and os.path.getsize(pickle_file_path) > 0:
        print(f"  Cache found for '{enzyme_name}'. Attempting to load...")
        try:
            with open(pickle_file_path, 'rb') as f:
                return pickle.load(f)
        except (EOFError, pickle.UnpicklingError) as e:
            print(f"  Error loading from corrupted/empty pickle file for '{enzyme_name}' ({e}). Will re-fetch data.")
            return None
        except Exception as e:
            print(f"  Unexpected error loading pickle for '{enzyme_name}': {e}. Will re-fetch data.")
            return None
    return None

#gets swiss prot record of each enzyme and also retries if there is connection issue
def get_swissprot_record_with_retry(uniprot_accession, max_retries=3):

    record = None
    for attempt in range(max_retries):
        try:
            with ExPASy.get_sprot_raw(uniprot_accession) as handle:
                record = SwissProt.read(handle)
            break
        except Exception as e:
            print(f"    Error fetching SwissProt for {uniprot_accession} (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                record = None
    return record

#what parses and saves catalytic info *THIS IS WHAT I THINK NEEDS ADJUSTMENT* because it will take only the first substrate and product when there might be multiple
def extract_catalytic_info_from_record(record):

    substrates = "Not found"
    products = "Not found"

    for comment in record.comments:
        if comment.startswith("CATALYTIC ACTIVITY:"):
            catalytic_activity = comment.split("CATALYTIC ACTIVITY:")[1].strip()


            if '=' in catalytic_activity:
                parts = catalytic_activity.split('=', 1)


                substrates_str = parts[0].strip()

                substrates_str = re.sub(r'^Reaction:\s*', '', substrates_str, flags=re.IGNORECASE).strip()


                products_str = parts[1].strip()

                products_str = re.split(r'[;\[]', products_str, 1)[0].strip()

                substrates = [s.strip() for s in substrates_str.split('+') if s.strip()]
                products = [p.strip() for p in products_str.split('+') if p.strip()]

            else:

                substrate_match = re.search(r"Substrate:\s*(.+?)(?=\s*Product:|\.|\s*\[|$)", catalytic_activity, re.DOTALL)
                product_match = re.search(r"Product:\s*(.+?)(?=\.|\s*\[|$)", catalytic_activity, re.DOTALL)

                if substrate_match:
                    substrates_str = re.sub(r'\s+', ' ', substrate_match.group(1)).strip()
                    substrates = [s.strip() for s in substrates_str.split(',')]

                if product_match:
                    products_str = re.sub(r'\s+', ' ', product_match.group(1)).strip()
                    products = [p.strip() for p in products_str.split(',')]

            break

    return substrates, products

#This is the main function for a single enzyme. It first tries to load from the cache.
#If it fails it queries the UniProt search API for a unique UniProt ID, then uses that ID to fetch the full Swiss-Prot record, extracts the catalytic info, and finally saves the result to a pickle file.
def get_uniprot_data_for_enzyme(enzyme_name):
    """
    Main function to fetch or load data for a single enzyme.

    Args:
        enzyme_name (str): The name of the enzyme to process.

    Returns:
        dict: A dictionary containing the enzyme's catalytic info, or None if not found.
    """
    print(f"\n--- Processing Enzyme: '{enzyme_name}' ---")

    cached_data = load_from_pickle(enzyme_name, PICKLE_DIR)
    if cached_data is not None:
        return cached_data

    print(f"  No valid cache found for '{enzyme_name}'. Fetching from UniProt.")
    try:

        query_url = "https://rest.uniprot.org/uniprotkb/search"
        params = {
            "query": f'"{enzyme_name}"',
            "format": "json",
            "size": 1
        }

        response = requests.get(query_url, params=params)
        response.raise_for_status()
        data = response.json()

        record_id = None
        if data.get('results') and len(data['results']) > 0:
            record_id = data['results'][0]['primaryAccession']

        if not record_id:
            print(f"  - No UniProt ID found for '{enzyme_name}'. Skipping.")
            return None

        print(f"  - Found UniProt ID: {record_id}")


        record = get_swissprot_record_with_retry(record_id)

        if record:
            substrates, products = extract_catalytic_info_from_record(record)

            enzyme_data = {
                "Enzyme": enzyme_name,
                "Substrates": substrates,
                "Products": products
            }


            save_to_pickle(enzyme_data, enzyme_name, PICKLE_DIR)
            return enzyme_data
        else:
            print(f"  - Could not retrieve full record for '{enzyme_name}'. Appending a placeholder.")
            return {
                "Enzyme": enzyme_name,
                "Substrates": "Not found",
                "Products": "Not found"
            }

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data for '{enzyme_name}' from UniProt: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while processing '{enzyme_name}': {e}")
        return None

# creates excel file with all collected data corresponding to that enzyme.
def create_output_excel(all_data, output_filename=OUTPUT_EXCEL_FILE):
    if not all_data:
        print("No data collected to write to Excel. Exiting.")
        return

    print(f"\nCreating output Excel file: '{output_filename}'...")
    df = pd.DataFrame(all_data)


    df['Substrates'] = df['Substrates'].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    df['Products'] = df['Products'].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

    df.to_excel(output_filename, index=False)
    print("Output Excel file created successfully.")

#This function ties all the other functions together. It checks for the Google Drive directory, creates a mock Excel file if needed, loads the enzyme names, loops through each enzyme to fetch its data using get_uniprot_data_for_enzyme, and finally calls create_output_excel to save the results.
def main():
    print(f"Checking for Google Drive directory at: {PICKLE_DIR}")
    if not os.path.exists(GOOGLE_DRIVE_MOUNT_PATH):
        print(f"Error: Google Drive path '{GOOGLE_DRIVE_MOUNT_PATH}' does not exist.")
        print("Please ensure your Google Drive is mounted in Google Colab before running the script.")
        return
    else:
        print("Google Drive directory found.")
    if not os.path.exists(INPUT_EXCEL_FILE):
        create_mock_excel(INPUT_EXCEL_FILE)

    try:
        enzyme_names = load_enzyme_names(INPUT_EXCEL_FILE, ENZYME_NAME_COLUMN)
    except ValueError as ve:
        print(f"Configuration Error: {ve}")
        return

    all_results_list = []
    for i, name in enumerate(enzyme_names):
        results_for_name = get_uniprot_data_for_enzyme(name)

        if i == 0 and results_for_name:
            print("\n--- Substrates and Products for the First Enzyme ---")
            print(f"Enzyme: {results_for_name['Enzyme']}")
            print(f"Substrates: {results_for_name['Substrates']}")
            print(f"Products: {results_for_name['Products']}")
            print("---------------------------------------------------\n")

        if results_for_name:
            all_results_list.append(results_for_name)
        time.sleep(1)

    create_output_excel(all_results_list, OUTPUT_EXCEL_FILE)

    print("\nProcess completed successfully!")

if __name__ == "__main__":
    main()

Checking for Google Drive directory at: /content/drive/MyDrive/Catalytic Pickles
Google Drive directory found.
Reading enzyme names from 'RNA Modification in Viruses.xlsx'...
Found 475 unique enzymes to process.

--- Processing Enzyme: 'P32579' ---
  No valid cache found for 'P32579'. Fetching from UniProt.
  - Found UniProt ID: P32579
  Data for 'P32579' saved to /content/drive/MyDrive/Catalytic Pickles/P32579.pkl

--- Substrates and Products for the First Enzyme ---
Enzyme: P32579
Substrates: ['Reaction']
Products: ['L-threonine', 'hydrogencarbonate', 'ATP = L-   threonylcarbamoyladenylate', 'diphosphate', 'H2O']
---------------------------------------------------


--- Processing Enzyme: 'P36999' ---
  No valid cache found for 'P36999'. Fetching from UniProt.
  - Found UniProt ID: P36999
  Data for 'P36999' saved to /content/drive/MyDrive/Catalytic Pickles/P36999.pkl

--- Processing Enzyme: 'P41833' ---
  No valid cache found for 'P41833'. Fetching from UniProt.
  - Found UniProt ID

Counting species per domains for each pfam

In [ ]:
import os
import glob
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/PFAM TSV"
output_file = "/content/drive/MyDrive/pfam_domain_counts.xlsx"


target_domains = ["Eukaryota", "Bacteria", "Viruses", "Archaea"]

summary_data = []

print(f"looking for TSV files in: {folder_path}")
tsv_files = glob.glob(os.path.join(folder_path, "*.tsv"))

if not tsv_files:
    print(f"no TSV files found in {folder_path}!")
else:
    print(f"Found {len(tsv_files)} files. Starting processing...")

    for file_path in tsv_files:
        filename = os.path.basename(file_path)

        # Extract PFAM ID using regex
        match = re.search(r"(PF\d+)", filename)
        pfam_id = match.group(1) if match else filename

        try:
            # Load only the necessary column
            df = pd.read_csv(file_path, sep='\t', usecols=['Taxonomic lineage'])

            # Normalize to lowercase
            lineages = df['Taxonomic lineage'].dropna().astype(str).str.lower()

            # Hash Map for counts (O(1) access)
            counts = {domain: 0 for domain in target_domains}

            # Linear Scan (O(N))
            for lineage in lineages:
                for domain in target_domains:
                    if domain.lower() in lineage:
                        counts[domain] += 1

            # Store results
            row = {'Pfam ID': pfam_id}
            row.update(counts)
            summary_data.append(row)

            print(f"Processed {pfam_id}")

        except ValueError:
            print(f"Skipping {filename}: 'Taxonomic lineage' column not found.")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    if summary_data:
        result_df = pd.DataFrame(summary_data)

        # Reorder columns
        cols = ['Pfam ID'] + target_domains
        result_df = result_df[cols]

        # Save to Drive
        result_df.to_excel(output_file, index=False)
        print("-" * 30)
        print(f"Success! Processed {len(summary_data)} files.")
        print(f"File saved to your Drive at: {output_file}")
    else:
        print("No data processed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
looking for TSV files in: /content/drive/MyDrive/PFAM TSV
Found 38 files. Starting processing...
Processed PF08704
Processed PF00145
Processed PF00258
Processed PF00749
Processed PF01134
Processed PF01163
Processed PF01171
Processed PF01207
Processed PF01266
Processed PF01416
Processed PF01596
Processed PF01702
Processed PF01715
Processed PF01728
Processed PF01746
Processed PF01926
Processed PF02475
Processed PF02547
Processed PF02926
Processed PF03054
Processed PF03481
Processed PF03966
Processed PF04013
Processed PF04055
Processed PF04179
Processed PF04816
Processed PF06175
Processed PF08489
Processed PF12934
Processed PF13418
Processed PF13484
Processed PF13621
Processed PF13649
Processed PF14437
Processed PF15387
Processed PF17884
Processed PF18389
Processed PF00586
------------------------------
Success! Processed 38 files.
File saved to your Drive at: /

this code and the next few cells for the other domains, uses the pathway code sheet to link enzyme pfam organisms to each pathway and defines if it is complete or not. If an organism conatins an enzyme it will record the pfam and uniprot ID of that enzyme.

for virus

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_viral_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Viral results
    output_folder = os.path.join(base_path, 'Viral_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # DSA Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Viral analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Viral_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Viruses
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Viruses"
        v_mask = primary_df[lineage_col].str.contains('Viruses', case=False, na=False)
        unique_viruses = sorted(primary_df[v_mask]['Organism'].unique())

        if not unique_viruses:
            pd.DataFrame({"Status": ["No Viruses Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for v_name in unique_viruses:
            v_lower = v_name.strip().lower()
            row_data = {"Organism": v_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if v_lower in org_lookup:
                                entry_id = org_lookup[v_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Exponential backoff/sleep to protect API
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Viral analysis
generate_viral_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Viral_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Viral analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Viral_Pathway_Analysis_Outputs


for archaea

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_archaea_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Archaea results
    output_folder = os.path.join(base_path, 'Archaea_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Archaea analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Archaea_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Archaea
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Archaea"
        a_mask = primary_df[lineage_col].str.contains('Archaea', case=False, na=False)
        unique_archaea = sorted(primary_df[a_mask]['Organism'].unique())

        if not unique_archaea:
            pd.DataFrame({"Status": ["No Archaea Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for a_name in unique_archaea:
            a_lower = a_name.strip().lower()
            row_data = {"Organism": a_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if a_lower in org_lookup:
                                entry_id = org_lookup[a_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Exponential backoff/sleep to protect API
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Archaea analysis
generate_archaea_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Archaea_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Archaea analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Archaea_Pathway_Analysis_Outputs


for Eukaryotes

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_eukaryotic_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Eukaryotic results
    output_folder = os.path.join(base_path, 'Eukaryotic_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory (crucial for large Eukaryotic TSVs)
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Eukaryotic analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Eukaryotic_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Eukaryotes
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Eukaryota"
        e_mask = primary_df[lineage_col].str.contains('Eukaryota', case=False, na=False)
        unique_eukaryotes = sorted(primary_df[e_mask]['Organism'].unique())

        if not unique_eukaryotes:
            pd.DataFrame({"Status": ["No Eukaryotes Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for e_name in unique_eukaryotes:
            e_lower = e_name.strip().lower()
            row_data = {"Organism": e_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if e_lower in org_lookup:
                                entry_id = org_lookup[e_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Protect API rate limits
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Eukaryotic analysis
generate_eukaryotic_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Eukaryotic_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Eukaryotic analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Eukaryotic_Pathway_Analysis_Outputs


code for bacteria

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_bacterial_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Bacterial results
    output_folder = os.path.join(base_path, 'Bacterial_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Bacterial analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Bacterial_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Bacteria
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Bacteria"
        b_mask = primary_df[lineage_col].str.contains('Bacteria', case=False, na=False)
        unique_bacteria = sorted(primary_df[b_mask]['Organism'].unique())

        if not unique_bacteria:
            pd.DataFrame({"Status": ["No Bacteria Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for b_name in unique_bacteria:
            b_lower = b_name.strip().lower()
            row_data = {"Organism": b_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if b_lower in org_lookup:
                                entry_id = org_lookup[b_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Protect API rate limits
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Bacterial analysis
generate_bacterial_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Bacterial_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Bacterial analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Bacterial_Pathway_Analysis_Outputs


Code checking if all the pfam files are in the PFAM TSV folder

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def scan_for_missing_pfams():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')

    # 2. Load Spreadsheet Data
    try:
        spreadsheet = gc.open_by_key(spreadsheet_id)
        # Pull Pfam IDs from both mapping and enzyme list
        gene_pfam_df = pd.DataFrame(spreadsheet.worksheet("Gene pfam").get_all_records())
        enzyme_info_df = pd.DataFrame(spreadsheet.worksheet("enzyme info").get_all_records())

        # Combine all unique Pfam IDs required for the project
        required_pfams = set()
        required_pfams.update(gene_pfam_df['pfam ID'].astype(str).str.strip().unique())
        required_pfams.update(enzyme_info_df['pfam ID'].astype(str).str.strip().unique())

        # Remove any empty strings or 'N/A'
        required_pfams = {p for p in required_pfams if p and p.lower() != 'nan' and p.lower() != 'n/a'}

    except Exception as e:
        print(f"Error accessing Google Sheet: {e}")
        return

    # 3. Index Existing Files in Drive
    # We look for the Pfam ID inside your versioned filenames (uniprotkb_PFXXXXX_date.tsv)
    print(f"Scanning folder: {pfam_folder}...")
    existing_files = glob.glob(os.path.join(pfam_folder, "*.tsv"))

    # Create a set of IDs that we actually have on disk
    found_pfam_ids = set()
    for fpath in existing_files:
        fname = os.path.basename(fpath)
        parts = fname.split('_')
        if len(parts) > 1:
            found_pfam_ids.add(parts[1].strip())

    # 4. Identify the Delta (Missing IDs)
    missing_pfams = sorted(list(required_pfams - found_pfam_ids))

    # 5. Output Results
    print("\n" + "="*30)
    print(f"SCAN COMPLETE")
    print(f"Total Unique Pfams Required: {len(required_pfams)}")
    print(f"Total Files Found in Drive: {len(found_pfam_ids)}")
    print(f"MISSING PFAMS: {len(missing_pfams)}")
    print("="*30)

    if missing_pfams:
        print("\nPlease download the TSV files for the following IDs:")
        for mp in missing_pfams:
            print(f" - {mp}")

        # Optional: Save this to a text file for easy copy-pasting
        with open(os.path.join(base_path, 'missing_pfams_list.txt'), 'w') as f:
            for mp in missing_pfams:
                f.write(f"{mp}\n")
        print(f"\nList saved to: {base_path}/missing_pfams_list.txt")
    else:
        print("\nAll required Pfam TSVs are present in your folder!")

# Run the scanner
scan_for_missing_pfams()

Mounted at /content/drive
Scanning folder: /content/drive/My Drive/Lauren Paprocki/PFAM TSV...

SCAN COMPLETE
Total Unique Pfams Required: 71
Total Files Found in Drive: 66
MISSING PFAMS: 10

Please download the TSV files for the following IDs:
 - Not Found
 - P25325
 - PF00383
 - PF00588
 - PF01189
 - PF01206
 - PF02403
 - PF04077
 - PF04358
 - PF13532

List saved to: /content/drive/My Drive/Lauren Paprocki/missing_pfams_list.txt


code to open seed file

In [ ]:
!pip install biopython
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

input_file = "PF00258.alignment.seed"
output_file = "PF00258_sequences.fasta"

clean_records = []

with open(input_file, "r") as in_handle:
    for record in SeqIO.parse(in_handle, "stockholm"):
        # Convert to string to safely remove gaps, then back to a Seq object
        sequence_string = str(record.seq).replace("-", "").replace(".", "")

        # Create a new record without the gaps
        new_record = SeqRecord(
            Seq(sequence_string),
            id=record.id,
            description=record.description
        )
        clean_records.append(new_record)

# Write all cleaned records to a FASTA file
with open(output_file, "w") as out_handle:
    SeqIO.write(clean_records, out_handle, "fasta")

print(f"Success! Created {output_file} with {len(clean_records)} sequences.")

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

Code fetching FASTA sequences from Uniprot IDs

In [ ]:
import requests
import time

# Paste your raw ID list here
raw_data = """
PF00258, A0A6A5XRE7
PF00258, A0ABD1V4T6
PF00258, A0A8B8JQF3
PF00258, A0A168RZW1
PF00258, A0A1X2IUX7
PF00258, A0AAE1KJ81
PF00258, L8HGF7
PF00258, A0A8B7Z8U5
PF00258, A0A091NBT3
PF00258, A0A498ST25
PF00258, A0A3Q1FIP1
PF00258, A0A9P0LPN2
PF00258, A0A812D275
PF00258, A0A316Z198
PF00258, A0A9N9JIZ9
PF00258, A0A8B9RPA5
PF00258, A0AAD5J8Q8
PF00258, A0AA39RQA5
PF00258, A0A5C7H9M4
PF00258, A0A6G1SKQ7
PF00258, A0AAN7C6Q3
PF00258, A0A1V9ZE56
PF00258, A0A6J2ANI1
PF00258, A0AAD8GD54
PF00258, A0A444UGC6
PF00258, A0AAF1BWT8
PF00258, A0AAV9FCX0
PF00258, A0AAV9BGZ6
PF00258, A0AAW2ZR25
PF00258, A0A455R5H4
PF00258, A0A914E098
PF00258, A0A7K7PTB0
PF00258, A0AAQ3RCD8
PF00258, A0A836G5H1
PF00258, F4WII3
PF00258, A0A836JYG4
PF00258, A0A836JD89
PF00258, A0AAD9Q7J1
PF00258, K8ESW6
PF00258, A0A6P8IBS8
PF00258, A0A2R6R2X5
PF00258, A0A7J0EK05
PF00258, A0A9P6QMT6
PF00258, A0A8R2JSP9
PF00258, A0A9D4V587
PF00258, A0A816FJD1
PF00258, A0A820S6Y2
PF00258, A0A0P6JS90
PF00258, A0A1W7R8G0
PF00258, R7WAN8
PF00258, A0A453LMH6
PF00258, A0A850Y6F8
PF00258, A0A7K6TU48
PF00258, A0A8H7C416
PF00258, K5X1V2
PF00258, A0A7K7KEH1
PF00258, A0A1W7RHX5
PF00258, A0A7F5RJH9
PF00258, A0A9W8MZS8
PF00258, A0A8H4VJV7
PF00258, D2HU79
PF00258, A0A8A1MIG9
PF00258, C0NSV8
PF00258, C6HF01
PF00258, F0UN90
PF00258, F2TRJ8
PF00258, A0A168KA03
PF00258, A0A9W8USX1
PF00258, A0A7L2BSW5
PF00258, A0A024GHD1
PF00258, F0WKL4
PF00258, A0A8T2MQN2
PF00258, A0A8T3CUW4
PF00258, A0A7K6ZBH7
PF00258, A0AAD7T4S5
PF00258, A0A7K8HBU0
PF00258, A0A8H3PIN1
PF00258, A0A7L0WTZ8
PF00258, A0A7S2IY65
PF00258, A0A7S1S9Q6
PF00258, A0A7S4TC35
PF00258, A0A8J2LG66
PF00258, A0A151P443
PF00258, A0A3Q0GPR2
PF00258, A0A0L0SUS2
PF00258, A0AAV6G7B5
PF00258, A0A4Q4NER6
PF00258, A0A4Q4SIL0
PF00258, A0A8J2IA66
PF00258, D1MX90
PF00258, A0A8H7EC02
PF00258, A0AAD4NW06
PF00258, A0AB37WHZ7
PF00258, A0A0C2XES0
PF00258, A0A2A9P0C5
PF00258, A0A0Q3X7N0
PF00258, A0A8B9G9Q2
PF00258, A0A7L0MQT8
PF00258, C4TP54
PF00258, A0A9N8ZME4
PF00258, A0A9N9H6H0
PF00258, A0AAQ4EZQ3
PF00258, A0A1E1XCH7
PF00258, A0A023FK14
PF00258, G3MG20
PF00258, A0A023GM55
PF00258, W1NF31
PF00258, A0AAD5G9J0
PF00258, A0A9W6Z0S3
PF00258, A0A7J6A1L8
PF00258, Q6QNI0
PF00258, A0A6A5WAZ0
PF00258, A0A7S0CXR0
PF00258, A0A2T3B8E4
PF00258, A0A6A5QWM7
PF00258, A0A6A4WII1
PF00258, A0A3Q0QXE1
PF00258, A0A1X7TVE0
PF00258, A0AAQ5YUW0
PF00258, A0A3P8U5B5
PF00258, A0A7S3P9A8
PF00258, A0A9P7YRM1
PF00258, A0A3N0XV80
PF00258, A0A7N6BFJ4
PF00258, A0AAV8AEF6
PF00258, A0A9Q0RC87
PF00258, A0A1Y1XEE6
PF00258, A0A199VDN6
PF00258, A0A6V7QF36
PF00258, R0L3J0
PF00258, A0A493T711
PF00258, A0A8B9ZPJ5
PF00258, A0A368H221
PF00258, A0A0D6MAY2
PF00258, A0A0C2GDV7
PF00258, A0A1S6LKK9
PF00258, A0A158P8B9
PF00258, A0A0R3PW88
PF00258, A0A7G2CA85
PF00258, A0A9D3MJV6
PF00258, A0A851PVC7
PF00258, A0A7L3GLK7
PF00258, A0A0M3KA06
PF00258, A0A9Q1MB43
PF00258, A0AAE1SEH4
PF00258, A0A803TIL3
PF00258, A0A8W7K8D9
PF00258, T1EAM7
PF00258, A0A1Y9GKC1
PF00258, A0AAG5DQI5
PF00258, A0A2M3ZJ00
PF00258, A0A182K283
PF00258, A0A8W7PZM8
PF00258, A0A182M2N1
PF00258, W5JI64
PF00258, A0A218NGQ1
PF00258, A0A182PFV5
PF00258, A0A182QMS6
PF00258, D9IQN0
PF00258, A9PL55
PF00258, F5HJH4
PF00258, A0A182T810
PF00258, A0A2M4BI19
PF00258, A0A182UH37
PF00258, A0A182V798
PF00258, D9IQN2
PF00258, Q6RXW9
PF00258, A0A1Y9J085
PF00258, A0A084WQX7
PF00258, D9IQN5
PF00258, D9IQN8
PF00258, A0A2M4AUE8
PF00258, V5H651
PF00258, C3KGV1
PF00258, A0A1P8Z7V1
PF00258, A0A8B9CUJ5
PF00258, A0A8B9IM22
PF00258, A0A1S5V8T1
PF00258, A0A7K9VMU9
PF00258, A0A7G6J4I4
PF00258, A0A6M3BDA0
PF00258, A0A7L2EJV7
PF00258, A0AAI8V8A5
PF00258, A0A1D1ZI05
PF00258, A0A4S4MR03
PF00258, A0A094N5L8
PF00258, A0A2K5F1B2
PF00258, A0A091N6I8
PF00258, A0AAW1TGP6
PF00258, A0AAW1RS94
PF00258, W4HB70
PF00258, A0A6G0XW86
PF00258, A0A418AHU9
PF00258, A0A6A4ZLI1
PF00258, A0A7K7CDB8
PF00258, A0A834Y103
PF00258, A0A6G0YLS2
PF00258, A0A6G0TZ96
PF00258, A0A9P0NHD9
PF00258, A0AA40K3Z1
PF00258, A0AAW0RCC1
PF00258, A0A427XKB0
PF00258, V9IBI4
PF00258, A0A2A3EKD6
PF00258, A0A7M7LSZ3
PF00258, A0A140B158
PF00258, A0A7S3UZN6
PF00258, A0A6A6BM12
PF00258, Q964N5
PF00258, A0AAE0I590
PF00258, A0A8S9XYT9
PF00258, A0A8H7BVX0
PF00258, A0A2I0AJZ5
PF00258, A0A087R0V5
PF00258, A0A8J4P865
PF00258, A0A8B7JZG3
PF00258, A0A8B9QHR7
PF00258, A0A2G9R496
PF00258, A5HVZ6
PF00258, A0AAN7SP37
PF00258, A0A663F4B5
PF00258, A0A2G5C4D4
PF00258, A0A8S2ASS7
PF00258, D7MY40
PF00258, A0A8T2AZG4
PF00258, Q56XF4
PF00258, A0A8T2GX34
PF00258, A0A087HIK7
PF00258, A0A565C8Q1
PF00258, A0A9C6TCK5
PF00258, A0A445DLA9
PF00258, A0A7L1T1F9
PF00258, A0A4Y2WBV8
PF00258, A0A0D6QUQ6
PF00258, Q9U4A9
PF00258, A0A6B2L032
PF00258, A0A8S1BJK3
PF00258, A0A7K8KT78
PF00258, A0A7L0I8W8
PF00258, A0A8T0FQT8
PF00258, A0A7R9RM61
PF00258, A0A0B6YVM6
PF00258, A0AAV7FDL0
PF00258, A0A5N5T0P6
PF00258, A0AA39N170
PF00258, A0A2H3DSM7
PF00258, A0AA39UX00
PF00258, A0AA39UEL9
PF00258, A0A284S0N7
PF00258, A0A2H3C6M7
PF00258, A0AA39KDK8
PF00258, A0AAV8Z897
PF00258, A0AA88KVB0
PF00258, A0A2U1QLP2
PF00258, A0AAN8RNB0
PF00258, A0AAD6IW30
PF00258, A0A437AAH2
PF00258, A0AAV9WCF9
PF00258, G1X1X8
PF00258, D4B2V4
PF00258, E5R298
PF00258, C5FK09
PF00258, A0AAN8A9I8
PF00258, A0A7K7L4I3
PF00258, A0AAU7PJ03
PF00258, A0A482WBW8
PF00258, A0A0M3IGD9
PF00258, F1L239
PF00258, A0A3N4I6A0
PF00258, A0A8H7IZF1
PF00258, A0A4S2N469
PF00258, A0A1D2VLA9
PF00258, A0A167YN15
PF00258, A0A5P1FKM9
PF00258, A0A1L9WYX3
PF00258, A0A5N6YI42
PF00258, A0A5N6U6A2
PF00258, A0A401L9C0
PF00258, A0A5N7BPF2
PF00258, A0A1F8ACW3
PF00258, A0A9W6DPJ7
PF00258, A0A1L9U706
PF00258, A0A5N6ZT08
PF00258, A0A0U5G2D9
PF00258, A0A2I1DEJ0
PF00258, A0A2I2FJL9
PF00258, A0A1R3RDI3
PF00258, A0A7R7VSU9
PF00258, A1CU36
PF00258, A0A5N6ZIN5
PF00258, A0A1E3BHU7
PF00258, A0A319DQT3
PF00258, A0A317UT18
PF00258, A0A8H6V1J9
PF00258, A0A8G1VVS3
PF00258, A0AB74CN39
PF00258, A0A7U2R0V7
PF00258, A0A8H4HHL3
PF00258, A0A229YCS6
PF00258, Q4X224
PF00258, B0XW88
PF00258, A0A1L9VM13
PF00258, A0A317WSM1
PF00258, A0A8H6QDI8
PF00258, A0A395HX63
PF00258, A0A395HBT8
PF00258, A0A2V5HWL4
PF00258, A0A8T8X367
PF00258, A0A7R8A543
PF00258, G7XMT1
PF00258, A0AAN5YG66
PF00258, A0A5N5WTA6
PF00258, A0A1M3TSP1
PF00258, A0A5N6JAY8
PF00258, A0A3D8SX67
PF00258, A0AAD4CT01
PF00258, A0A318ZHJ0
PF00258, A0AAJ8E1B3
PF00258, G3XX93
PF00258, A2QKK3
PF00258, A0A370C643
PF00258, A0A0L1IUY2
PF00258, A0A2I1CLX0
PF00258, A0A5N6EUK8
PF00258, A0A0F8V1R6
PF00258, A0A2T5M3Z2
PF00258, A0AAN4YUW5
PF00258, I8TTF6
PF00258, Q2U1M5
PF00258, A0A5N6E2S4
PF00258, A0A0F0IB44
PF00258, A0A5C0C9J5
PF00258, A0A370PTP8
PF00258, A0A8G1VPT4
PF00258, A0A5N7DJ87
PF00258, A0A5N6T3G5
PF00258, A0A9P3BMD0
PF00258, A0A7R8AK28
PF00258, A0A0F8X9V7
PF00258, A0A017SK52
PF00258, A0A319AUG2
PF00258, A0A3A2ZA67
PF00258, A0A319EHN3
PF00258, A0A317XEP9
PF00258, A0A5N6XIZ8
PF00258, A0A2I2G2E8
PF00258, A0A1L9TSR2
PF00258, A0A2J5HFM1
PF00258, A0A5N6V498
PF00258, A0A5M9MI83
PF00258, A0A5M3YNE2
PF00258, Q0CAE3
PF00258, A0A397HWZ8
PF00258, A0A5N6WGL4
PF00258, A0A8H3XX92
PF00258, A0A1L9N713
PF00258, A0A421CYN8
PF00258, A0A8H3RRB1
PF00258, A0A319D9K0
PF00258, A0A319BRG7
PF00258, A0A1L9PWL3
PF00258, A0A2V5H525
PF00258, A0A9P3BSN6
PF00258, A0A3F3Q971
PF00258, A0A1L9RST1
PF00258, Q3YE95
PF00258, A0AAX7UXP2
PF00258, A0A7S0KX22
PF00258, A0A9P7GDJ3
PF00258, A0AAD3HGM7
PF00258, W5LN15
PF00258, A0A167VW38
PF00258, A0A663MXR9
PF00258, A0A7L3X2Q6
PF00258, A0A8J7PBL7
PF00258, A0A852PCC0
PF00258, A0A158NWJ0
PF00258, A0A195BVA0
PF00258, A0A7S2XL59
PF00258, A0A6G1HC86
PF00258, A0A9P8KC05
PF00258, A0A074VFZ4
PF00258, A0A9N8PL38
PF00258, A0A074WY83
PF00258, A0AB38M4V6
PF00258, A0A074Y1U0
PF00258, A0A074YDA3
PF00258, A0A9N8KDP1
PF00258, A0A9N8JFJ7
PF00258, F0YFP9
PF00258, A0A7S3K3K1
PF00258, J0DCY0
PF00258, A0AAX4HER7
PF00258, A0A2I4AQQ6
PF00258, A0A9Q3EMD7
PF00258, A0A1D2A3F3
PF00258, A0A6J3E6L1
PF00258, A0A088CNH0
PF00258, S4UD24
PF00258, A0A061D633
PF00258, A7ARX0
PF00258, A0AAV4LTG2
PF00258, A0AAD9LMD8
PF00258, A0AAD9UQ65
PF00258, A0AAD8LR78
PF00258, A0A0K3ARY8
PF00258, A0A2H6KBT5
PF00258, A0A9W5TA78
PF00258, A0A1E3QZ59
PF00258, A0A6I9VMS2
PF00258, A0A0K8VX13
PF00258, A0A556TXJ6
PF00258, A0A7L2UBN6
PF00258, A0A384AWB7
PF00258, A0A8B8XNL4
PF00258, A0A6A1QB49
PF00258, A0A087VFH3
PF00258, A0A2P4THP9
PF00258, A0A7K9EF37
PF00258, A0A1Y1Z7X0
PF00258, K8F527
PF00258, A0ABD0LJ90
PF00258, F4NSN0
PF00258, A0A177WX32
PF00258, M2MY98
PF00258, A0AAW0RLB1
PF00258, A0A2S7Y469
PF00258, J4UWV8
PF00258, A0A0A2W1Q8
PF00258, A0A167FVT2
PF00258, A0A9P0F0R6
PF00258, A0A2A9MQ58
PF00258, A0A9W2XRC9
PF00258, A0A7S1CE24
PF00258, A0A6J1NRT4
PF00258, A0A261Y262
PF00258, A0A6A5V4C1
PF00258, A0A9W3B893
PF00258, A0AAD8F3Y0
PF00258, A0A8H7NEB1
PF00258, W6ZG33
PF00258, W7EGC2
PF00258, A0A6P3J8K7
PF00258, A0A499T5E9
PF00258, A0A060TI78
PF00258, D8MAN7
PF00258, V9VGW6
PF00258, A0A179UPE1
PF00258, A0A2B7XMC5
PF00258, A0A1J9PZG7
PF00258, A0A0H1BFZ5
PF00258, A0AAU9K6Y2
PF00258, A0A9Q0M024
PF00258, N1JAG1
PF00258, A0A9W4GCQ5
PF00258, A0A9X9L857
PF00258, A0A381LIV5
PF00258, A0A383UZK4
PF00258, A0A4P9VZ20
PF00258, A0A0S4KLI2
PF00258, A0AAD4C5S6
PF00258, A0A8I3AEX7
PF00258, A0AA39XMS2
PF00258, A0A6P8NFU0
PF00258, A0A6P3V1X4
PF00258, A0A9C6SHK7
PF00258, A0A6J3L7C3
PF00258, A0A7L1MH75
PF00258, A0FGR6
PF00258, A0A8R2C5A4
PF00258, A0A4S4LZS9
PF00258, A0AAD5Y601
PF00258, A0A6P5DMY3
PF00258, A0A4W2H7N1
PF00258, L8HZY4
PF00258, A0A8C0A8Z8
PF00258, A0AAA9TQV4
PF00258, A0A6B2F2Y4
PF00258, A0A6B2F5G9
PF00258, A0A067MLY9
PF00258, A0A6J3UET9
PF00258, A0A8H4N1U8
PF00258, R1GKV0
PF00258, A0A4Y8CMY5
PF00258, A0A4Z1I1Z0
PF00258, Q6H9H8
PF00258, A0A384JZW5
PF00258, M7UV93
PF00258, G2YKH0
PF00258, A0A4Z1HQT9
PF00258, A0A9P5M859
PF00258, A0A4Z1JUV3
PF00258, A0A8H6EMQ6
PF00258, A0A4S8QW75
PF00258, A0A4Z1H9X4
PF00258, A0A4Z1FVK4
PF00258, A0A4Z1L4R4
PF00258, A0A4Z1ESQ8
PF00258, A0A813M3M2
PF00258, A0A3M7P1P5
PF00258, A0A0Q3R8X8
PF00258, A0A7K7MBE6
PF00258, A0A7L2VWV6
PF00258, A0A6V7LA33
PF00258, A0A6P5AW57
PF00258, C3XSJ5
PF00258, A0A8J9VAE8
PF00258, A0A398ATH9
PF00258, A0A8X7WK44
PF00258, A0A8S9S702
PF00258, A0ABQ8E3M8
PF00258, A0A3P6G3A0
PF00258, A0A0D3DLN5
PF00258, A0A9P0FI42
PF00258, A0A976IHJ0
PF00258, A0A8J9Y825
PF00258, A0A448YSB8
PF00258, A0A4E9F7G8
PF00258, A0A0N4TR13
PF00258, A0A0R3QBV8
PF00258, A0A8C0FNQ0
PF00258, A0A7K9I6Y5
PF00258, A0A091I2X7
PF00258, A0A7K4YPL7
PF00258, A0AAV6WSS0
PF00258, A0A977JB77
PF00258, A0A7J7KI56
PF00258, A0A7L3HH36
PF00258, A0A7K4TGP5
PF00258, A0A811L6U2
PF00258, A0A1I7RXZ3
PF00258, A0A8B9Z4L6
PF00258, A0A443HJV0
PF00258, V5FYC9
PF00258, A0A6A5U377
PF00258, A0A8D9AFV0
PF00258, A0A8H7T3P1
PF00258, A0A9P1MVU8
PF00258, A0A8S1HHK0
PF00258, A0A8S1EWM7
PF00258, G0NEB6
PF00258, A8WZA4
PF00258, Q9N535
PF00258, A0A8R1IT80
PF00258, A0A261BNR2
PF00258, A0A2G5UVT4
PF00258, E3NV23
PF00258, A0A1I7TWM0
PF00258, A0AAV4UZ80
PF00258, A0AAV4TIU1
PF00258, A0A7S0JQG2
PF00258, A0A8C3GNP7
PF00258, A0A151RXF9
PF00258, A0A852AYB5
PF00258, A0A7S0NVF3
PF00258, A0AAV2TXY4
PF00258, A0A8C3PTL2
PF00258, C1C308
PF00258, A0A7L4LKA0
PF00258, A0A226NB23
PF00258, A0A068F5Y9
PF00258, U3FE10
PF00258, V9L9V0
PF00258, A0A3Q7NPW0
PF00258, A0A653BNX4
PF00258, A0A165E2F4
PF00258, A0A167NI09
PF00258, A0A7K6T7Q8
PF00258, A0A7L3XSR6
PF00258, A0A977P477
PF00258, A0A9P7Z756
PF00258, A0A091IQ91
PF00258, A0A851CNP4
PF00258, A0A7J7HTJ4
PF00258, A0A4S4ECL9
PF00258, A0A9W3HKR0
PF00258, A0A5N4EF42
PF00258, A0A8B8SYA4
PF00258, E2ATB8
PF00258, A0A7G8Z812
PF00258, A0A851N335
PF00258, A0AAN6TAK2
PF00258, A0AAN9PNT3
PF00258, A0A8H6BWT6
PF00258, A0A1D8PPU8
PF00258, C4YSR9
PF00258, A0AB34Q0Y0
PF00258, H2ER12
PF00258, A0A9W6WJK3
PF00258, B9WIU1
PF00258, A0A0W0CVA4
PF00258, Q6FXR3
PF00258, P50126
PF00258, M3K4H0
PF00258, A0A8H7ZJB5
PF00258, H8X8I8
PF00258, A0AAI9SU09
PF00258, A0A8X7NHT9
PF00258, G8B526
PF00258, G3BAG0
PF00258, A0AAD5BFJ4
PF00258, Q66T17
PF00258, C5M8U5
PF00258, A0A9W4U0M8
PF00258, A0A367YM70
PF00258, A0AB36W826
PF00258, A0A2V1AAK7
PF00258, A0A2V1AY86
PF00258, A0A2P7YRK3
PF00258, A0A8S3ZP48
PF00258, A0A4Q2DSB4
PF00258, A0A9W8MJW0
PF00258, A0A8C0R5Y7
PF00258, A0A8I3RTE2
PF00258, A0AAQ3KBN9
PF00258, A0A7J6DNT7
PF00258, R7UUZ3
PF00258, A0A8C2SL75
PF00258, W9YP88
PF00258, W9Z535
PF00258, A0A0D2X5H8
PF00258, R0I197
PF00258, A0A2G2ZN29
PF00258, A0A2G2WYI7
PF00258, Q6DUB3
PF00258, Q2PQU5
PF00258, D5I3H5
PF00258, A0ABD1BZ71
PF00258, A0A7K5MTR8
PF00258, A0AAW2GYX2
PF00258, A0A833VEE5
PF00258, A0A091MNI8
PF00258, A0A1U7T4Z0
PF00258, A0A9Q1GN91
PF00258, A0A8J6BDD6
PF00258, A0A5N6QMG8
PF00258, A0A922EPH1
PF00258, A0A8J4R6L2
PF00258, A0ABD3DE82
PF00258, A0A8C0WYG7
PF00258, A0A7K8NQU3
PF00258, A0A8C3YF28
PF00258, A0A1Y2I5U1
PF00258, Q05001
PF00258, A0A091LFR3
PF00258, A0A7L2DE25
PF00258, A0A8C3US87
PF00258, A0A4P9XA11
PF00258, F4PW62
PF00258, Q9WUN7
PF00258, A0A2K5S104
PF00258, A0AA38TVT7
PF00258, A0A499SXZ9
PF00258, Q6PLI6
PF00258, A0A499SZ96
PF00258, A0A499T6A4
PF00258, A0A499SWU5
PF00258, A0A499SZA9
PF00258, A0A499T5G6
PF00258, A0A499SXQ6
PF00258, A0A499T2W5
PF00258, A0A499SXR0
PF00258, A0A499T8N9
PF00258, A0A499SWT6
PF00258, A0A499SXS0
PF00258, A0A499SY05
PF00258, A0A499T6B6
PF00258, A0A499SY32
PF00258, A0A499SWT1
PF00258, A0A499T6C2
PF00258, A0A499T8R2
PF00258, A0A852MH71
PF00258, A0A7K5ABG8
PF00258, A0A7K5U936
PF00258, A0AAE8N5R2
PF00258, A0A1Q3C5V5
PF00258, A0AAJ7BZY2
PF00258, A0A7L3S572
PF00258, A0A0N7L8X9
PF00258, A0A316VRD8
PF00258, A0AAJ8E5E3
PF00258, W8C881
PF00258, A0A5N5QFV5
PF00258, A0A2C5X7A3
PF00258, A0A0F8BUF7
PF00258, A0A8T0HHB4
PF00258, A0A8T2QCN9
PF00258, A0AAJ7E2M2
PF00258, A0A2K5P688
PF00258, A0AA40CN99
PF00258, A0AA39ZKG1
PF00258, A0AAE0M9A5
PF00258, A0A8J2MUZ0
PF00258, A0A2S6BXP5
PF00258, A0A2G5HPZ1
PF00258, A0A9P3FMT8
PF00258, A0A6A6FPZ2
PF00258, A0A4Y7LWZ6
PF00258, M2RAM8
PF00258, A0AAW0FF71
PF00258, A0A7L1WK85
PF00258, A0A851SBJ2
PF00258, A0A212CYK1
PF00258, A0A9N8VJK0
PF00258, A0A7L3QML1
PF00258, A0A7L4JLI8
PF00258, A0A9N9MKP3
PF00258, A0A7L4N0J8
PF00258, A0A7S3Q5U7
PF00258, A0AAD3HC59
PF00258, A0AAN6VT23
PF00258, A0AAE0HDW8
PF00258, Q2H128
PF00258, A0AAJ0M2P8
PF00258, G0S6C6
PF00258, A0A7L3DYC9
PF00258, A0A7K8JCZ6
PF00258, A0AAN8CLX6
PF00258, A0AAN8CFA7
PF00258, A0A6G1QMI8
PF00258, A0AA88NM06
PF00258, A0A6J2UZ60
PF00258, A0A388LYE4
PF00258, A0A0A0AFW7
PF00258, A0A7L0K380
PF00258, M7BYX4
PF00258, A0A8C0G8K2
PF00258, A0A8T1S1C2
PF00258, A0A803NBT9
PF00258, A0AAW0WIA8
PF00258, R4UB73
PF00258, A0A401S3W1
PF00258, A0A8C2V5F3
PF00258, A0A7K7FKD1
PF00258, A0A8J4YDD9
PF00258, A0A9P0IU82
PF00258, A0A7R9Z7Z0
PF00258, A0A250X271
PF00258, A0A835TR21
PF00258, A0A7S0WZH6
PF00258, A8IDE3
PF00258, A0A835WU01
PF00258, A0A091KZT0
PF00258, A0A3L8T160
PF00258, A0AAD5DFY4
PF00258, A0A2P6TZA7
PF00258, E1ZA31
PF00258, A0A9D4TZ07
PF00258, A0A0D9RZX8
PF00258, A0A7K9U6B2
PF00258, A0A7S3E478
PF00258, A0A7S2T1H7
PF00258, A0AAX4PLW5
PF00258, A0A852BIV6
PF00258, A0A850V292
PF00258, A0A1C7MVL2
PF00258, A0A3N4K475
PF00258, R7QSN6
PF00258, A0A7L0UJR9
PF00258, A0A7K5P0G8
PF00258, A0A0G4FWH7
PF00258, A0A6T5WBT4
PF00258, A0A8C3FC56
PF00258, A0A9B0WKE0
PF00258, A0A0M0K9P5
PF00258, A0A9P0FW49
PF00258, A0A8C3Q3G7
PF00258, A0A5B9RUS1
PF00258, A0AAD7XQ04
PF00258, A0A7S4F1C5
PF00258, A0A7K5GUR2
PF00258, A0A507F0Y1
PF00258, A0A7K8VFE0
PF00258, A0A3Q7Y0D5
PF00258, A0ABD2Q4H6
PF00258, A0A7L0B7P1
PF00258, A0A8I6RV65
PF00258, A0A5E4NH87
PF00258, A0ABD2ZA75
PF00258, A0A7L2JFG7
PF00258, A0A3S3NNS2
PF00258, F6TWQ7
PF00258, H2Z1M7
PF00258, A0A7L4A8X6
PF00258, A0A8H7SEJ7
PF00258, A0AA88P500
PF00258, A0ABD0N3Z8
PF00258, A0A7L1R2H1
PF00258, V4UHZ5
PF00258, A0A067GZ30
PF00258, A0A2H5NNL7
PF00258, A0AAP0MYZ3
PF00258, A0A9P1FXR6
PF00258, A0AA39V8H9
PF00258, A0A5C1Z9B3
PF00258, A0A0D2FJK4
PF00258, A0A1C1D0Z6
PF00258, V9D4I3
PF00258, A0AA39CQK8
PF00258, A0A0D2CVD9
PF00258, W9X2C9
PF00258, W9VRJ7
PF00258, A0AAV9HFD0
PF00258, A0AB34KLD1
PF00258, A0A024FS65
PF00258, A0A8J5C990
PF00258, A0A1B6CFF3
PF00258, A0A6A5SX50
PF00258, A0AAV5AMM9
PF00258, A0A9P7QDE8
PF00258, A0A8K0J9I6
PF00258, A0A9P7SSW0
PF00258, A0A9P7Q6Y3
PF00258, A0A9P7MC95
PF00258, M1VX10
PF00258, A0A9P7SXH1
PF00258, A0AA91Q2S3
PF00258, C4YAE7
PF00258, A0A7K6QSU4
PF00258, A0AAN9K520
PF00258, A0A8S1CBC5
PF00258, A0A1Y1ZEZ1
PF00258, G7YRK5
PF00258, A0A9N9Y642
PF00258, A0AA35Q8G2
PF00258, A0A9N9VAL1
PF00258, A0A9P0EGZ1
PF00258, A0A1J1ITU3
PF00258, A0A8M1KRS4
PF00258, A0AAD5U275
PF00258, A0A7M5USV6
PF00258, A0A1S5ZY34
PF00258, A0A7K8AHL2
PF00258, A0AA40HNN9
PF00258, J3K376
PF00258, A0A0J8S2K6
PF00258, A0A0J6Y7E0
PF00258, A0A0J8RBQ1
PF00258, C5P685
PF00258, E9DIV9
PF00258, A0A0J6FCB1
PF00258, A0A7S0LTA8
PF00258, I0YY96
PF00258, A0AAV1IGE7
PF00258, A0A7K8PQQ8
PF00258, W6YAW2
PF00258, N4XNZ2
PF00258, M2TDC5
PF00258, B1NWC7
PF00258, A0A8H6DRC9
PF00258, M2SR97
PF00258, A0A8K0N958
PF00258, A0A9W8IQL6
PF00258, A0A9W8CJ55
PF00258, A0A9W7Y9W3
PF00258, A0A9W8IGC7
PF00258, A0A9W8CQV1
PF00258, A0A9W8LSE9
PF00258, A0A9W8LG46
PF00258, A0A9W8LGS6
PF00258, A0A9W8GU20
PF00258, A0A2G5BG64
PF00258, A0A9W8L1D8
PF00258, A0A9W8EI13
PF00258, A0A6P6TKR2
PF00258, A0A068UIR7
PF00258, A0ABD1JDZ9
PF00258, A0A3D8S2T7
PF00258, A0A3D8SG73
PF00258, A0A091KUP0
PF00258, A0A9P9XNJ9
PF00258, A0A8H3W522
PF00258, A0A1Q8RX03
PF00258, A0AAD9EQJ7
PF00258, A0AAI9YVD4
PF00258, A0AAI9Y503
PF00258, A0AAX4J2H0
PF00258, A0A010RGK7
PF00258, L2GHV6
PF00258, A0A8H4FPF1
PF00258, T0LR72
PF00258, A0AAJ0EZ87
PF00258, E3QXW1
PF00258, A0A4T0WEA5
PF00258, A0A1B7XTP4
PF00258, A0A162PCD6
PF00258, A0AAD9YII2
PF00258, A0A9P6I062
PF00258, A0AA37GEH4
PF00258, A0A9Q8WIR9
PF00258, A0AAI9V3I9
PF00258, A0A8H6MI18
PF00258, A0AAD8PJF7
PF00258, A0A9W4RS03
PF00258, A0A135TEF3
PF00258, N4VWR9
PF00258, A0A1G4BLI9
PF00258, A0AAI9ZZ39
PF00258, A0A8H6NJL0
PF00258, A0A135V882
PF00258, A0A9P7UDQ7
PF00258, A0A5Q4C637
PF00258, A0A9P5BS72
PF00258, A0A4R8TIN8
PF00258, A0A135RX27
PF00258, A0A8H6MZQ7
PF00258, A0AA37PB32
PF00258, A0A4R8QTD3
PF00258, A0A066XLC5
PF00258, A0AAV9TW26
PF00258, A0A4U6X515
PF00258, A0A166YHX0
PF00258, A0A4R8RP59
PF00258, A0AAD9H4L5
PF00258, A0A4U5VJ33
PF00258, A0A9P6CKK0
PF00258, A0A8H3U0I6
PF00258, A0A8H5HPM3
PF00258, A0A0D0C038
PF00258, A0A2K5J5Y1
PF00258, A0A843XSK9
PF00258, R7VWI4
PF00258, A0A7K4SAY9
PF00258, A0A9Q1DSP3
PF00258, A0A137P1B2
PF00258, A0A2T3A0K9
PF00258, A0AA38VZI0
PF00258, A0A1J7JQR8
PF00258, A0A420YM85
PF00258, A0A5M3MQ40
PF00258, R7Z2W8
PF00258, A0AAJ0FVN6
PF00258, A0A4Y7TJF8
PF00258, D6RQ83
PF00258, A0A5C3L3I2
PF00258, A0A851VNM8
PF00258, A0A835M3W3
PF00258, A0A6L2PUI8
PF00258, A0A1R3JY63
PF00258, A0A1R3I3X7
PF00258, A0A179IES0
PF00258, A0A167UAB3
PF00258, A0A545VAS5
PF00258, A0A2H4SM70
PF00258, G3JE72
PF00258, A0AAN8QE23
PF00258, U5EZ84
PF00258, A0A7S1FZ01
PF00258, A0A140B168
PF00258, A0A091EZ11
PF00258, A0A8U7N5Z4
PF00258, A0AAN7HNW3
PF00258, A0A2T2NHK8
PF00258, A0A221C8V9
PF00258, A0A851LS92
PF00258, A0A7L0FWI4
PF00258, A0A8J2H9Z8
PF00258, A0AAV7I3T2
PF00258, A0A8J5VDZ8
PF00258, A0A6J2RSK1
PF00258, A0A8C2YDC6
PF00258, A0A7R9WRA9
PF00258, A0A8B8E949
PF00258, A0AAV9SLC0
PF00258, A0A9P6JJA9
PF00258, G3IH71
PF00258, A0A8K0XUX6
PF00258, A0A7M4G0S9
PF00258, A0A4D6Q737
PF00258, A0A6G1APB0
PF00258, A0A9P6TH33
PF00258, A0AAN9IHW3
PF00258, A0AAW1BNV9
PF00258, A0A0K8RT07
PF00258, A0A0A8KYK4
PF00258, A0A7K5HV90
PF00258, A0A5C3ME33
PF00258, A0A1V8SMX4
PF00258, A0A4V5NJL0
PF00258, A0A9P5CRQ3
PF00258, A0A1E3HFA5
PF00258, A0A1E3JNL5
PF00258, A0A0D0VIN1
PF00258, Q5K8J1
PF00258, A0AAJ8JUR9
PF00258, A0A095C1A8
PF00258, A0A0D0V7Z4
PF00258, A0A5D3AV16
PF00258, E6R7K5
PF00258, J9VR96
PF00258, A0A854QEG0
PF00258, P0CP13
PF00258, P0CP12
PF00258, A0A1E3IRZ9
PF00258, A0A7U0TIC7
PF00258, A0ABD2MQC0
PF00258, A0A7S0QSV4
PF00258, A0A1J4MU08
PF00258, A0A9D5DG18
PF00258, A0A0S4TG54
PF00258, A0A2P4Z0U6
PF00258, B6AHS8
PF00258, A0A7S7LHA2
PF00258, Q5CQZ6
PF00258, A0A1J4MDC5
PF00258, A0AAV9Y0I9
PF00258, A0A2J7R7T8
PF00258, A0A7K4KSJ4
PF00258, A0A7K4LVB4
PF00258, E7E1L4
PF00258, A0A091GND1
PF00258, A0A9I9D064
PF00258, A0A5A7TNB7
PF00258, A0A0A0LPA4
PF00258, A0AAV6NL85
PF00258, A0A6J1K9S7
PF00258, A0A6J1HDZ5
PF00258, A0A9P4GKJ6
PF00258, A0A8H4RRP1
PF00258, A0A1B6G5R2
PF00258, A0A8D8MQJ2
PF00258, A0A385L4D0
PF00258, A0ABD1CWU3
PF00258, B0XKF9
PF00258, A0A1Q3F924
PF00258, A0A336MC07
PF00258, A0AAW2AMX2
PF00258, Q9P4E1
PF00258, Q9P4E2
PF00258, A0A061QHR2
PF00258, A0A9Q8Z4T0
PF00258, A0A9P4THZ4
PF00258, A0A328E6J0
PF00258, A0A484L9A4
PF00258, A0AAV0CLW9
PF00258, A0A9P0Z974
PF00258, A0AA48QY66
PF00258, A0A0J1BDJ4
PF00258, A0AAD3YDI3
PF00258, A0A7J7IR06
PF00258, M1VB21
PF00258, A0AAV9IR57
PF00258, A0A8C0ZDX8
PF00258, A0A8C3P565
PF00258, A0A1V2L9P7
PF00258, A0A1E4S886
PF00258, A0A8S0XNV6
PF00258, A0A7S1D7Y3
PF00258, A0A8C2Z9G8
PF00258, A0A6P6RRY6
PF00258, A0ABD3SG98
PF00258, A0ABD3QWE7
PF00258, A0ABD3QNL5
PF00258, A0A7U1BJ65
PF00258, A0AA36M408
PF00258, A0A3P6RJD1
PF00258, A0A0D7B215
PF00258, A0A9P5LKG9
PF00258, A0AAD2FVS1
PF00258, A0AAE0LLD0
PF00258, A0A103Y5C7
PF00258, A0A3P8VQK5
PF00258, A0A0N1HBK7
PF00258, W2S5I7
PF00258, A0A195CZ09
PF00258, A0A7R8ZUY8
PF00258, A0A3Q2CYR7
PF00258, A0A9Q9YUL7
PF00258, A0A9J8D2V3
PF00258, Q9C498
PF00258, A0A2C6L6I3
PF00258, A0A423WQ43
PF00258, A0A423XEE0
PF00258, A0A194VWV7
PF00258, A0AAN9YLW5
PF00258, A0A423X173
PF00258, M5GGB8
PF00258, S8CD77
PF00258, A0A9P9EHE5
PF00258, A0A9P9JFG6
PF00258, A0A165QTG6
PF00258, A0AAX6MYI7
PF00258, A0A8J2VTJ9
PF00258, A0A212F8K1
PF00258, A0A8M2B9H1
PF00258, A0A553RNX8
PF00258, I1VC15
PF00258, A0A4Y7M133
PF00258, A0A4Y7M1V2
PF00258, A0A4Y7LYZ4
PF00258, A0A4Y7M1C3
PF00258, A0A8J2WN32
PF00258, A0A4Y7M6G7
PF00258, A0A4Y7M9E4
PF00258, A0A4Y7MBF6
PF00258, A0A0P4YES8
PF00258, I1VC16
PF00258, I1VC22
PF00258, A0A4Y7MXJ9
PF00258, A0A4Y7N4M7
PF00258, A0AAD5PT02
PF00258, A0A7K6FM71
PF00258, A0A7R9FSV4
PF00258, A0A7K6HWC2
PF00258, A0ABD2DH44
PF00258, A0A140B171
PF00258, A0AAF0X9L9
PF00258, A0A5B7BJ08
PF00258, P42059
PF00258, A0A0V1PZG0
PF00258, Q6BVA3
PF00258, A0A6A5KT86
PF00258, A0AAP0GWR9
PF00258, A0A8H6B6Z0
PF00258, A0A9P4N2C8
PF00258, A0A2Y9P9V8
PF00258, A0A2I0WU75
PF00258, A0AAV7GIJ3
PF00258, A0A8T3BR35
PF00258, A0ABD0VHL0
PF00258, A0A0M3R7Y6
PF00258, U4UAJ3
PF00258, A0A4S8MAE8
PF00258, A0A9P9IT72
PF00258, A0AAY4D7U0
PF00258, A0A4Y9Z4R4
PF00258, A0A9N9DET8
PF00258, A0A9D4P6R2
PF00258, A0A6P6XNY5
PF00258, K9J3I3
PF00258, A0A9X0CX24
PF00258, A0A9P0GXN2
PF00258, A0A6P7F6C8
PF00258, A0A8J6CFZ9
PF00258, A0A3Q0JAB1
PF00258, A0A0G2FX27
PF00258, A0A2P5IF85
PF00258, A0A9N9W742
PF00258, A0AAN9UUB5
PF00258, A0A3P7L5N0
PF00258, A0A7K9JZQ1
PF00258, A0A8C4F1L9
PF00258, A0A7J7EFR8
PF00258, A0A1E5WAJ1
PF00258, A0A4Q9MC88
PF00258, R7SLX8
PF00258, A0AAN6ZNF9
PF00258, A0A7L0A8X9
PF00258, A0A0D8Y250
PF00258, Q55CT1
PF00258, A0AAN7TZ25
PF00258, F0Z982
PF00258, A0A6A5RMW3
PF00258, A0A9W8X234
PF00258, A0A9P5BX10
PF00258, A0A9W9D764
PF00258, A0A163EVV2
PF00258, A0A8S2YMP3
PF00258, A0A9W9C5B6
PF00258, A0A835AAK9
PF00258, A0AAN8VSM8
PF00258, A0A4V1J5I0
PF00258, A0A9W8B4H9
PF00258, G8E4B8
PF00258, A0A7I8VBS2
PF00258, A0A6P3XWT3
PF00258, A0A443RNS4
PF00258, A0AB40CUT7
PF00258, A0A9D5HR56
PF00258, A0AA38LVF2
PF00258, A0A218ZDZ1
PF00258, A0AAD9SVM2
PF00258, A0A1J9RAJ8
PF00258, A0A1S8B7D1
PF00258, A0AAN6S8V4
PF00258, A0AAD8AD16
PF00258, A0A2A2KRN0
PF00258, A0A1S3EXX9
PF00258, A0AAD9XIK0
PF00258, A0AAE0EKE2
PF00258, Q8MU49
PF00258, A0ABD3N509
PF00258, A0A9W8APZ9
PF00258, A0A6J3M1R6
PF00258, A0A9P6V0D4
PF00258, A0AAD9FFM1
PF00258, A0A7J5YH49
PF00258, A0AAD4R110
PF00258, A0A915DZ96
PF00258, A0A7S4WAD5
PF00258, A0A642UZM9
PF00258, A0A9N9FK86
PF00258, A0A397HFZ1
PF00258, A0A851JJJ1
PF00258, A0A2Z7B4I4
PF00258, A0A6A6AFV1
PF00258, N1PXR8
PF00258, A0AAV1SG91
PF00258, A0A0N4UKK2
PF00258, A0A151GQ76
PF00258, W7I007
PF00258, A0A9D4S409
PF00258, A0A8C4KM43
PF00258, A0A7K5XGS4
PF00258, A0A9C6T250
PF00258, B3MKG0
PF00258, A0A0M5IYG0
PF00258, A0A0Q5WB04
PF00258, B4JPU3
PF00258, A0A3B0JXX7
PF00258, A0A9P9YPI6
PF00258, A0A6J1M9H3
PF00258, A0A6P4IYI6
PF00258, A0A6J2U623
PF00258, A0AAU9FUH6
PF00258, A0A6P8LFD6
PF00258, Q961A7
PF00258, P91655
PF00258, A0A0Q9X5K3
PF00258, A0A484BVL3
PF00258, B4H635
PF00258, A0A6I8VZT0
PF00258, A0A6P4FMT1
PF00258, A0AAD4PJI6
PF00258, B4I1U1
PF00258, A0A0J9QXN1
PF00258, A0AB40D5E2
PF00258, B4LS74
PF00258, B4N4C3
PF00258, A0A0R1DJL4
PF00258, A0A7L3KYI3
PF00258, A0A093GHV6
PF00258, A0A851E783
PF00258, A0A154PJZ3
PF00258, A0A7S3VQD7
PF00258, A0A6P5ZTD7
PF00258, A0A665U1J6
PF00258, U6IZU1
PF00258, A0A068YC12
PF00258, A0A183B0U0
PF00258, A0AAJ0B3I0
PF00258, D8LBE3
PF00258, A0A835YFF9
PF00258, J9DJW9
PF00258, A0A7K9NNC6
PF00258, A0A875RWQ5
PF00258, A0AA36N7X4
PF00258, A0A091JJF0
PF00258, U6GZH1
PF00258, U6M302
PF00258, U6KZY8
PF00258, A0A851UNQ9
PF00258, A0A6I9RDE6
PF00258, A0A0R3RZ77
PF00258, A0A232M3K7
PF00258, A0AAN7W7B7
PF00258, A0AAY5F3V4
PF00258, A0AAD9DQI2
PF00258, A0AAN7WX53
PF00258, A0AAV5F5H7
PF00258, A0A8J6FRX8
PF00258, A0AAW1RG82
PF00258, A0A6A6FYZ1
PF00258, A0A4V6YAU6
PF00258, A0A8K0L7K3
PF00258, A0A3S0Z183
PF00258, A0AAE1B0I6
PF00258, A0AAV4J822
PF00258, A0A7K4VPE5
PF00258, A0A1B7NYG1
PF00258, A0A1J9PLW0
PF00258, Q5B0U2
PF00258, A0A7U0KIF9
PF00258, A0A9P7ZVG7
PF00258, A0A9Q0BIB5
PF00258, R1FNA7
PF00258, A0A0D3KMX6
PF00258, A0A2B7Z9Y4
PF00258, A0A0G2JA69
PF00258, A0AAF0IHL0
PF00258, M1JJ00
PF00258, Q8SS06
PF00258, A0A9Q9C7Z3
PF00258, E0S742
PF00258, I6ZIA5
PF00258, A0A8H7E548
PF00258, U1FY69
PF00258, A0AAV7BHT7
PF00258, A0A2Y9K4W9
PF00258, A0A426XZG9
PF00258, B0ERU0
PF00258, S0B0Z9
PF00258, C4M5R3
PF00258, N9V0I3
PF00258, M3TTF0
PF00258, M7X0C1
PF00258, M2S9N6
PF00258, Q2PCA9
PF00258, A0A0A1UDJ0
PF00258, Q2PCA8
PF00258, K2HRZ5
PF00258, Q2PCA6
PF00258, A0A0N4V8L4
PF00258, A0A1W0E670
PF00258, A0A1Y1S8N5
PF00258, A0A7S3DWB7
PF00258, A0A9P6MWK0
PF00258, A0A9P3LYW9
PF00258, A0A851XX70
PF00258, E6Y366
PF00258, B1PBA2
PF00258, A0A7S9KM27
PF00258, A0A1Y2ME69
PF00258, A0A8C4WV04
PF00258, A0A8C4MIY7
PF00258, A0A8C4PIW4
PF00258, A0A9L0TUQ2
PF00258, A0A5J9T397
PF00258, A0A6G1FX88
PF00258, G8JP40
PF00258, Q755C2
PF00258, A0A0X8HTU6
PF00258, A0A1S3W829
PF00258, A0A0K2CTD2
PF00258, A0A7K7GIR0
PF00258, A0A8C4RSV5
PF00258, A0A7L2Y5E4
PF00258, A0ABC8KLF9
PF00258, A0A420HTL8
PF00258, A0A2S4PNQ5
PF00258, A0A022RYS4
PF00258, A0A7K5PQL3
PF00258, A0A7S1XIX2
PF00258, A0A7S0XJ07
PF00258, A0AAV8T5Q3
PF00258, A0AA88VW73
PF00258, A0AA88QYE9
PF00258, A0A3S6JH60
PF00258, A0AB34I5N2
PF00258, O24425
PF00258, A0A0M9VVW4
PF00258, A0AAY5KV76
PF00258, A0A5J5CGE6
PF00258, A0AA97LLK9
PF00258, A0A4Y7LKT5
PF00258, A0A7K8XQN1
PF00258, A0ABD3L2T7
PF00258, A0A059ADZ7
PF00258, A0A7S2WMK9
PF00258, A0A7K7VGD8
PF00258, A0A8J4NS54
PF00258, A0A8K0FQ13
PF00258, A0A8J4NAD2
PF00258, A0A8J4NLE6
PF00258, A0A8S9ESM8
PF00258, A0A310SCX9
PF00258, Q94IN5
PF00258, A0A7K8CUR2
PF00258, A0A4C1TZH3
PF00258, A0AAU9UFF9
PF00258, A0A7S3NZR9
PF00258, A0A1Y3BHR5
PF00258, A0A093IJL5
PF00258, A0A7L4DB46
PF00258, E4MW95
PF00258, V4KCW7
PF00258, A0A7S4GK31
PF00258, M7T190
PF00258, A0A9N6WRI0
PF00258, A0A499T6B1
PF00258, A0A913YEH7
PF00258, A0A165HPD1
PF00258, A0AAV8VX14
PF00258, A0A072P199
PF00258, A0AAV9NQ91
PF00258, A0AAN6EZJ9
PF00258, H6C3Q0
PF00258, A0A438NHY7
PF00258, A0A0D2DM46
PF00258, A0A0D1XBE7
PF00258, A0A0D2A9P5
PF00258, A0AAN6E4J0
PF00258, A0A0D2FF24
PF00258, R0IQI2
PF00258, A0AAJ0LX84
PF00258, A0A7S3IB35
PF00258, A0A2N9GQK3
PF00258, A0A8C4UJB9
PF00258, A0A7K6LRC0
PF00258, A0A504Z1B6
PF00258, A0A2H1CSF0
PF00258, A0A8E0RVQ1
PF00258, A0AAW0D7K6
PF00258, A0A4Y1QK09
PF00258, A0A7S2V0K4
PF00258, J4H477
PF00258, U3JXH3
PF00258, A0AA88DKT6
PF00258, A0A8K0NNS8
PF00258, A0A1Z5KTE5
PF00258, A0A0D7A4C2
PF00258, A0ABD1MVZ3
PF00258, A0A7S2FP55
PF00258, A0A226D891
PF00258, S8E679
PF00258, A0A178ZKN5
PF00258, A0A177FJ24
PF00258, A0A0D2KJD1
PF00258, A0A178CQG8
PF00258, A0A0D2GUN4
PF00258, A0A058Z6E0
PF00258, A0A0C9QWM0
PF00258, A0A7L0NP74
PF00258, A0ABD1WSX0
PF00258, A0A1E7FTB4
PF00258, A0AAE1H725
PF00258, A0A9C6U3I3
PF00258, A0AAD2A353
PF00258, A0A850W5C7
PF00258, A0A7L3Z920
PF00258, A0AAN6KV03
PF00258, A0A4U0XYI3
PF00258, A0A833VMS8
PF00258, A0A091E764
PF00258, A0A093L224
PF00258, A0A3Q2QQK5
PF00258, A0A9N9H0K0
PF00258, A0A9W4WZV4
PF00258, A0A9N9I582
PF00258, A0A2T9Z4E6
PF00258, A0A7K5BGC0
PF00258, A0A8H4NKX1
PF00258, A0A9P5E896
PF00258, A0A8H4PGN5
PF00258, A0A428TAC0
PF00258, A0A8H5EB21
PF00258, A0A8H4K5A1
PF00258, A0AAN6CAR4
PF00258, A0A9P7KSS8
PF00258, A0A125RKB6
PF00258, A0A125RKB7
PF00258, A0A9P5ADE9
PF00258, A0A125RKB8
PF00258, A0A8H5UGI6
PF00258, A0A366SCS6
PF00258, A0A125RKB9
PF00258, A0A2T4HBX5
PF00258, A0A8H5XK34
PF00258, A0A125RKC0
PF00258, A0A428PU14
PF00258, A0A8J2NH15
PF00258, A0A430M1A6
PF00258, A0A9W8QZ93
PF00258, A0A395ML94
PF00258, A0A428RPP2
PF00258, A0A125RKC2
PF00258, A0A9Q9RWW8
PF00258, A0A8H4WMR3
PF00258, A0A8H5XZJ5
PF00258, A0A8H5WZC3
PF00258, A0A9W8UAJ3
PF00258, A0A3M2SID2
PF00258, A0A0N1J2P8
PF00258, A0A395T9L8
PF00258, A0A1L7T209
PF00258, A0A8H5MV82
PF00258, A0A125RKC4
PF00258, A0A8H5YEY5
PF00258, A0A9P8DLI6
PF00258, A0A8H5ICA9
PF00258, X0K7E1
PF00258, A0A428TF03
PF00258, A0A8H5EGI8
PF00258, A0A0D2XLL0
PF00258, W9JW73
PF00258, W9J8A6
PF00258, A0A3L6NRX7
PF00258, A0A8H6LKG9
PF00258, X0JR47
PF00258, A0A5C6TIC1
PF00258, N4U6P3
PF00258, N1R5X0
PF00258, A0A0S1LIK2
PF00258, A0A0J9WJP6
PF00258, X0B033
PF00258, A0A4Q2VYF3
PF00258, W9NQ40
PF00258, A0A2H3GPL5
PF00258, A0A8J5U7K6
PF00258, A0A8J5PSR7
PF00258, X0D6W2
PF00258, X0NQU3
PF00258, A0A8H5I3A4
PF00258, A0A9W8WKM2
PF00258, A0A1B8B428
PF00258, A0A1L7WBI1
PF00258, A0A8H5Q376
PF00258, A0A8H5PH63
PF00258, K3VYV6
PF00258, A0A9P9KKK4
PF00258, A0A8H4XDD2
PF00258, A0A125RKC8
PF00258, A0A9P9H3M5
PF00258, E5LC41
PF00258, A0A218PG36
PF00258, A0A125RKB3
PF00258, A0A395SL06
PF00258, A0A125RKD0
PF00258, A0A8H5V6W8
PF00258, A0A9W8V8M8
PF00258, A0AAE8M6C2
PF00258, A0A8K0S681
PF00258, C7ZFM6
PF00258, A0A2L2TF68
PF00258, A0A9P7L0N7
PF00258, A0A8H4U2S8
PF00258, A0A8C5CS05
PF00258, J3NR62
PF00258, A0A7K9T6Z5
PF00258, A0A9C7UTQ4
PF00258, M2Y9E4
PF00258, A0AAV9IMW7
PF00258, A0A8J6A1H5
PF00258, A0AAJ6VWP2
PF00258, A0A067THL3
PF00258, A0A6J1WXS4
PF00258, H9CZQ8
PF00258, A0A315VMB7
PF00258, A0A5K1JVJ2
PF00258, A0A2I6QP29
PF00258, A0A2G8SNF6
PF00258, G3PGQ0
PF00258, A0A093H4Y0
PF00258, Q6E5V9
PF00258, S8CSW0
PF00258, A0A2Z0N1Z6
PF00258, A0A7K4JHB1
PF00258, A0AA35W4H9
PF00258, A0A9P4YMK2
PF00258, A0A8N5F201
PF00258, A0A8C3MFS7
PF00258, A0A9P5KRZ6
PF00258, A0A6P8QEA6
PF00258, A0AAD5XSX8
PF00258, A0A132P0L2
PF00258, V6TTF8
PF00258, C6LS90
PF00258, E2RTY8
PF00258, E1F8T0
PF00258, A0A4Z1SL74
PF00258, S0E6G6
PF00258, A0A420U6H6
PF00258, E5LC59
PF00258, W7MPB4
PF00258, A0A2K0VTR3
PF00258, A0A125RKC7
PF00258, A0A8H5L6G7
PF00258, A0A125RKD1
PF00258, A0A4U9F3S7
PF00258, I1RCS6
PF00258, A0A8H4ASS6
PF00258, A0A397VGA2
PF00258, S3CRV2
PF00258, H0ERX6
PF00258, A0A7L4MGZ1
PF00258, A0A7L0S5W5
PF00258, K3X4C7
PF00258, A0A183BK54
PF00258, A0A914HET1
PF00258, Q8X172
PF00258, S7RDC3
PF00258, A0AAD8UIU1
PF00258, A0A397TPG1
PF00258, A0A8E2JLI4
PF00258, A0A1A9VG31
PF00258, A0A1A9WAM1
PF00258, A0A8U0W4H7
PF00258, D3TKP7
PF00258, A0A1B0A3W0
PF00258, A0A1B0BD66
PF00258, A0A9P8II27
PF00258, A0A0R0L4H1
PF00258, A0A445LSJ7
PF00258, A0A5P1KAY0
PF00258, A0ABD6ERE2
PF00258, A0A9W8Z4C4
PF00258, A0A8E3XNJ0
PF00258, A0A420IJ47
PF00258, A0A8H3IAU5
PF00258, A0A139ASP6
PF00258, A0A183DWH5
PF00258, A0A150H417
PF00258, A0A452GTA5
PF00258, A0A8C4WQB5
PF00258, G3S3A8
PF00258, A0A8J6DC26
PF00258, A0A0B0PST3
PF00258, A0A7J8WI27
PF00258, A0A7J9IKH0
PF00258, A0A5B6UNL2
PF00258, A0A5J5WVJ2
PF00258, A0A5D2FV65
PF00258, A0A7J8QUM5
PF00258, A0A7J9D233
PF00258, A0A7J9H4K3
PF00258, A0A1U8PDS8
PF00258, A0A7J8TP09
PF00258, A0A7J9A171
PF00258, A0A7J8L849
PF00258, A0A5D2WRC2
PF00258, A0A0D2RSM7
PF00258, A0A7J9KMR0
PF00258, A0A9D3WCL3
PF00258, A0A5D2RTY8
PF00258, A0A7J9DAP5
PF00258, A0A8C5N6C7
PF00258, A0A2V3J2J2
PF00258, A0A7K9ACJ9
PF00258, A0A7S1Y1N8
PF00258, A0A7K6E5R4
PF00258, A0A1B6MTM8
PF00258, A0A023B0T8
PF00258, A0A1C7MDQ3
PF00258, F0XGP3
PF00258, A0A850TQB6
PF00258, A0ABC9WCP4
PF00258, B7XH76
PF00258, A0AAN9Z861
PF00258, A0A7S4N2U7
PF00258, L1JII2
PF00258, A0A9X9M7K6
PF00258, A0A9P7VZJ3
PF00258, A0A6P8VFU3
PF00258, A0A409YTU5
PF00258, A0A9P5NWE7
PF00258, A0A6A4HYP8
PF00258, A0A7L1B9Y3
PF00258, A0A0L7REA1
PF00258, A0A1W7RA54
PF00258, A0A9J6GRE4
PF00258, A0A1L8E8H1
PF00258, A0A6A0AFN9
PF00258, A0A7I4Y448
PF00258, A0A158QQ66
PF00258, A0A851Z312
PF00258, A0A7K7NRH5
PF00258, A0A6C0SLC8
PF00258, A0A6C0PND5
PF00258, A0AAN9FUM5
PF00258, A0A8J8NUV1
PF00258, A0A4Q9LE91
PF00258, A0A4Q9LW63
PF00258, A0A2G9HWC3
PF00258, A0A1L0CPI9
PF00258, A0A1E5R673
PF00258, A0A1E5RGL7
PF00258, A0A1E5RD23
PF00258, A0A1B7T9T4
PF00258, A0A7S0F0H0
PF00258, A0A3Q2WC03
PF00258, A0A086TEN1
PF00258, A0A7S2N650
PF00258, A0A7S3BRL8
PF00258, E2BB88
PF00258, A0A0C3CCJ0
PF00258, A0A9K3EMI7
PF00258, Q96561
PF00258, A0A2B7YC29
PF00258, A0A7S2MJW1
PF00258, A0A2W1BS47
PF00258, A0A183G5C0
PF00258, A0A5C3MRX0
PF00258, A0A7L2BB39
PF00258, A0A2A4JG35
PF00258, T1G1Y7
PF00258, A0A7L1S0W9
PF00258, A0AAE0Q8X6
PF00258, A0A9D3NAD4
PF00258, A0A7K9DL95
PF00258, A0A646QEM1
PF00258, A0A7S1HPJ8
PF00258, A0A7S0YSH9
PF00258, A0A6G3MIY5
PF00258, I2H173
PF00258, A0AAW1U9S4
PF00258, A0AAD8N4G4
PF00258, R4QY78
PF00258, A0A4Y9ZK68
PF00258, A0A4S4KRA5
PF00258, A0A7R8YUI1
PF00258, A0A7L0GHP5
PF00258, A0A6J1AD52
PF00258, A0A1X2GVX7
PF00258, W4KHB7
PF00258, G5BIS2
PF00258, A0ABD2HT55
PF00258, A0ABD2KNJ8
PF00258, A0A8H3IQH5
PF00258, A0A1I7XKQ4
PF00258, A0A7S3YKH8
PF00258, D3BES8
PF00258, A0A6A6NB73
PF00258, A0AA86VDH2
PF00258, A0A6A3CQ46
PF00258, A0A9W7MRM8
PF00258, A0A7L1KNY6
PF00258, A0A3Q2YXU0
PF00258, A0A7L2M1P3
PF00258, A0A8B7RMR7
PF00258, A0A6A7FQY1
PF00258, A0A0F7ZNS7
PF00258, A0A9P8SK64
PF00258, A0A7L4F2K8
PF00258, A0A3M0KT75
PF00258, A0A9Q1CGE9
PF00258, A0A1B6IBM5
PF00258, A0A8J5TTM7
PF00258, H0Y9Q0
PF00258, A0A2R5GM25
PF00258, F2EH66
PF00258, A0A7L3MRY9
PF00258, A0A3M7AWW9
PF00258, A0A1Z5TTY1
PF00258, A0A4W5Q389
PF00258, J7S1R6
PF00258, A0A979FWE8
PF00258, A0A131XHP2
PF00258, M4C2B5
PF00258, A0AAV0V2J2
PF00258, A0A2J6TIH6
PF00258, A0A2J6QII1
PF00258, A0A2J6RJ93
PF00258, A0A0R3WLK7
PF00258, A0A0C9W9D2
PF00258, A0A9P6DUC4
PF00258, T2MGU1
PF00258, A0A7K5WQ85
PF00258, A0A8T2K5M7
PF00258, A0A564ZDW2
PF00258, A0A9N9M4M0
PF00258, A0A9N9PQE0
PF00258, Q6QZW3
PF00258, A0A161I4M3
PF00258, A0A9P7B1D8
PF00258, A0A0D2NJ12
PF00258, A0A1E4RQ14
PF00258, G9P1M0
PF00258, A0A024SI03
PF00258, G0RLM2
PF00258, G9MSC0
PF00258, A0A7L2QIV2
PF00258, A0A6G1RI55
PF00258, A0ABD1EJE0
PF00258, A0A9X6RKA8
PF00258, A0A369JMQ8
PF00258, A0A7K7U889
PF00258, G0QUE3
PF00258, A0A1A7Z1M6
PF00258, W5U6V5
PF00258, I3NG69
PF00258, A0A7K6P4G7
PF00258, A0A8K0D5B4
PF00258, A0ABC8V3S8
PF00258, A0A7L1C426
PF00258, A0AA39X4M8
PF00258, A0A8H3IVS1
PF00258, A0A7L1G5M6
PF00258, A0A0C5LBP0
PF00258, A0A177B7K0
PF00258, A0A7K9QUW7
PF00258, A0AAX6I0G8
PF00258, V5HK99
PF00258, A0A4D5RGE5
PF00258, A0A067QAC7
PF00258, A0A7L2Z4M9
PF00258, A0A8C5P411
PF00258, A0A316UU89
PF00258, A0A067L1Y3
PF00258, A0A433QJM1
PF00258, A0A833X9W2
PF00258, A0A8C5J9V0
PF00258, A0A7N0T972
PF00258, V5GGT6
PF00258, Q00GM4
PF00258, A0A9P4U8N3
PF00258, H2B064
PF00258, A0A7J7MGN1
PF00258, A0A9K3GNS7
PF00258, A0A1Y1HTQ2
PF00258, A0A0A8L650
PF00258, Q6CWI0
PF00258, W0TH92
PF00258, A0AAV2L728
PF00258, A0AAN8EST6
PF00258, A0AA39CV77
PF00258, A0A1Y1UF20
PF00258, A0A1B2JF06
PF00258, F2QV97
PF00258, C4R6G5
PF00258, A0A3Q3B5Q0
PF00258, W6MU50
PF00258, A0A1B9G6K4
PF00258, A0AAJ8MEF5
PF00258, A0AAX4K3Y6
PF00258, A0AAX4KQQ3
PF00258, A0A1B9GXK1
PF00258, A0A1B9J189
PF00258, A0AAW0YGG4
PF00258, A0AAJ8KZS8
PF00258, A0AAJ8LML8
PF00258, A0A0F7L1I9
PF00258, B2G2W4
PF00258, A0A3Q3FQE6
PF00258, A0A0C9XWN4
PF00258, B0DFI5
PF00258, A0A1G4K324
PF00258, A0A1G4M9E6
PF00258, A0A0C7MXQ3
PF00258, A0A1G4IUT0
PF00258, A0A1G4KAK3
PF00258, A0A1G4KK54
PF00258, A0A0P1KTJ6
PF00258, C5E2D4
PF00258, A0A8T9B1Y5
PF00258, A0A7D8UXJ4
PF00258, A0A8H8U2N0
PF00258, A0A8H8RXW2
PF00258, A0A8H8RDQ3
PF00258, A0A8T9CIL3
PF00258, A0A559M347
PF00258, A0AAD4QHL9
PF00258, A0AA35Y2Q4
PF00258, A0A9R1XWV5
PF00258, A0AAU9PG65
PF00258, A0A8K0KLY1
PF00258, A0A165DAC1
PF00258, A0AAV2YZ16
PF00258, A0A835NLM0
PF00258, A0A7K5RV37
PF00258, K4JMN0
PF00258, A0A6G0IW12
PF00258, A0AAV2BKL3
PF00258, A0A852H5P6
PF00258, A0A5M8Q5R5
PF00258, A0A9E8K095
PF00258, A0AA40D3S7
PF00258, A0A5N5DKP5
PF00258, A0AAJ0HJ13
PF00258, A0AA40DZB4
PF00258, A0AAE0KMP4
PF00258, A0AA40DTP1
PF00258, A0A0J7MYS3
PF00258, A0AAV2PC86
PF00258, A0AAJ8AX95
PF00258, A0AAD3R453
PF00258, A0A8C5WT87
PF00258, H3AFP6
PF00258, A0AAI8Z1J4
PF00258, A0A0D9WAH2
PF00258, Q7YXG3
PF00258, A0A7L1Z651
PF00258, A4HMS7
PF00258, A0A3P3ZH68
PF00258, E9BSD0
PF00258, A0A836H1N5
PF00258, A0A1E1J662
PF00258, A4IBG5
PF00258, A0AAW3AIU9
PF00258, E9AF79
PF00258, A0A836GA41
PF00258, E9B6D5
PF00258, A0AAW3BXG2
PF00258, A0A836GLV7
PF00258, A0A088SJ65
PF00258, A0AAW3BYX0
PF00258, A0A640KWG5
PF00258, A0AAW3ALT2
PF00258, A0A9W9DJQ6
PF00258, A0AA38NQM9
PF00258, A0AA38PZ15
PF00258, A0A1Q3DWB2
PF00258, A0AA38N5Z6
PF00258, A0A9W9DYQ3
PF00258, A0AA38PKT0
PF00258, A0A371D8J6
PF00258, A0A5C2SNH0
PF00258, A0A6G1JC87
PF00258, A0A7R8D8S3
PF00258, A0A8E2EL11
PF00258, A0A6J0I1A4
PF00258, W5N7W4
PF00258, A0AAD9ZDA5
PF00258, A0A5E4QFF6
PF00258, A0A7S8BFG2
PF00258, A0A8C5PAB0
PF00258, A0A7L0UT56
PF00258, A0A7S2L988
PF00258, A0A0N0E0A8
PF00258, A0A0N1IBR8
PF00258, A0A7F8R4W0
PF00258, A0AAV1IUV1
PF00258, A0A091Q9N5
PF00258, E4ZRA2
PF00258, A0A443RZU0
PF00258, A0A8H6G3N6
PF00258, A0A8H6FI85
PF00258, A0AAD5YYN6
PF00258, A0A8H5LM73
PF00258, A0A7K8ELC5
PF00258, A0A1Y2G2Y9
PF00258, A0A068RJV8
PF00258, A0AAD8DIJ8
PF00258, A0A077X3C9
PF00258, A0A2I0TCD3
PF00258, A0A1Y1WGT7
PF00258, A0A6A6P546
PF00258, A0A1S3I1Q8
PF00258, A0A197JQQ4
PF00258, A0AAD4DKM8
PF00258, A0A9P6RF50
PF00258, A0A9P7XM77
PF00258, A0A9P5RZV1
PF00258, A0AAV0MPI8
PF00258, A0AAV2EEX0
PF00258, A0A4Z2E8S9
PF00258, A0A1E3QCH5
PF00258, A0AAD7QPF6
PF00258, A0A340X0Q0
PF00258, A0AAP0NF61
PF00258, A0AAW2E825
PF00258, A0AAN7T3U4
PF00258, A0AAV3PT14
PF00258, A0A3P6T8A3
PF00258, A0AAN9FVB9
PF00258, A0A1S0TSS8
PF00258, A0A1Y2H1G6
PF00258, A0A2H4BK51
PF00258, A0A1L2EC44
PF00258, A0A6G5XH19
PF00258, A5E4M0
PF00258, A0A9P4KAT3
PF00258, A0AAD8R4P0
PF00258, A0A2N3N8L9
PF00258, A0A218V584
PF00258, A0A6A6TH24
PF00258, A0A6A5ZQ35
PF00258, A0A6A6R3A7
PF00258, A0A7K8KCS4
PF00258, A0A7S3YY13
PF00258, A0A7S2XE04
PF00258, V4AW26
PF00258, B5BSX2
PF00258, A0A7K7J7D1
PF00258, A0A7K9GXV8
PF00258, G3TPT4
PF00258, A0ABD0TGC2
PF00258, F5HS59
PF00258, A0A0L0BZL1
PF00258, A0A9P6KHV1
PF00258, A0A6A5NNR0
PF00258, A0A1J7HMA0
PF00258, A0AAV1Y6W8
PF00258, A0A7G3AQG8
PF00258, A0A146LNJ7
PF00258, A0AAV2IKC3
PF00258, A0A9N6WY89
PF00258, A0A667G492
PF00258, A0A485NM65
PF00258, A0A9P3UUC0
PF00258, Q4R9D6
PF00258, H9FJI1
PF00258, A0A2K6DWE8
PF00258, A0A7K6J9U6
PF00258, A0A200R7H0
PF00258, A0A2R2Q2B3
PF00258, A0A9P6BZQ9
PF00258, K2RDU3
PF00258, A0AAV0WWX5
PF00258, A0A267FWJ0
PF00258, A0A150ASP4
PF00258, K1PY38
PF00258, A0A0C4DPH4
PF00258, A0A5E8C327
PF00258, A0AAJ5YZE9
PF00258, A0AAF0DUT7
PF00258, A0AAF0E759
PF00258, A0AAF0EV05
PF00258, A0AAF0IYP7
PF00258, A8PW02
PF00258, A0AAF0F637
PF00258, A0AAF0EL05
PF00258, A0AAF0DXV5
PF00258, A0A0M8ML94
PF00258, A0AAF0FC27
PF00258, A0A3G2S442
PF00258, M5E4X4
PF00258, A0A2N1JEZ6
PF00258, A0AAJ5YPU9
PF00258, A0A8C5U4A0
PF00258, A0A7K6H2L0
PF00258, A0A540NEC4
PF00258, A0A498J571
PF00258, Q5YJK3
PF00258, A0A093QQZ7
PF00258, A0A2K5ZYT8
PF00258, A0A922CDT3
PF00258, A0A2C9UFV3
PF00258, A0A7S0X5V0
PF00258, A0A9P7RPP2
PF00258, A0A140CZM6
PF00258, A0AAF6ALA1
PF00258, A0A8C6ERN1
PF00258, A0A5E4DKJ4
PF00258, K1WQX3
PF00258, A0A6A6RNG7
PF00258, A0A7C8MA42
PF00258, A0A7N8WYJ9
PF00258, Q2PCA4
PF00258, X2FJ19
PF00258, A0A8H2VJH2
PF00258, A0A9P6W986
PF00258, A0AAV5S1W1
PF00258, A0A1X7R4D8
PF00258, A0A9D4B1V1
PF00258, A0A3P9D206
PF00258, G7L2T0
PF00258, A0A7R9LXC5
PF00258, A0A4Y7NGD2
PF00258, A0A075F9W8
PF00258, A0A9D3QJ96
PF00258, A0AAV7XEI4
PF00258, A0AAV2SL05
PF00258, T1GJT3
PF00258, A0A316VAA3
PF00258, F4RMT9
PF00258, A0A2H8TRV9
PF00258, A0A7K8A3C1
PF00258, A0A6A6XUJ5
PF00258, A0AAJ5C5K0
PF00258, A0A077RBP6
PF00258, H9H0V9
PF00258, A0A1X6Y3A2
PF00258, A0AA40G433
PF00258, A0A0N0BJ54
PF00258, A0A6V7WR61
PF00258, A0A915NYH5
PF00258, A0A8T0A2K8
PF00258, A0A1I8C107
PF00258, A0A914LXV8
PF00258, A0A915NAD6
PF00258, A0A8V5FS12
PF00258, A0A7K4Q7F6
PF00258, A0A8S4AW57
PF00258, A0AAW2H8I6
PF00258, A0AA97MNI4
PF00258, Q5W9H7
PF00258, A0AAN7YHV9
PF00258, A0AA47P7G6
PF00258, A0A091QTB4
PF00258, A0A7L0PRF9
PF00258, A0A091REW4
PF00258, A0A5K3EX69
PF00258, Q5F323
PF00258, A0AAF3J741
PF00258, A0AA36CRB7
PF00258, E9EHM0
PF00258, A0A0B2WXQ4
PF00258, A0A0B4FWZ0
PF00258, A0A0D9P1N4
PF00258, A0A7D5V352
PF00258, A0A0B4ICE1
PF00258, A0A9P8SCC5
PF00258, A0A5C6GJT6
PF00258, A0A0A1V446
PF00258, E9EYG6
PF00258, A0A4P6XPN3
PF00258, A0A4P9ZD45
PF00258, A0A1A0HK46
PF00258, A0A8H7GVA4
PF00258, A5DR76
PF00258, A0A2P6VQZ8
PF00258, A0A238FMM0
PF00258, U5HH46
PF00258, A0A2X0KXF4
PF00258, A0A2X0LQ45
PF00258, A0A6P7YG67
PF00258, A0A8C6EJE6
PF00258, A0AA39F961
PF00258, A0AA39KV62
PF00258, A0A136IWM3
PF00258, A0A9P9BJP3
PF00258, C1E5R0
PF00258, A0A7S0IG14
PF00258, C1N4X9
PF00258, A0A089Q756
PF00258, A0A6D2JTL5
PF00258, A0A6A6UK28
PF00258, A0A8J6G350
PF00258, A0A2H6NEW4
PF00258, A0A2D4G9F9
PF00258, U3ESB8
PF00258, A0A2D4HG84
PF00258, A0A2D4MB81
PF00258, A0A2D4Q0C3
PF00258, A0A5N6PXX6
PF00258, A0A2K8MN79
PF00258, A0A7S0P7Z6
PF00258, A0A7S0FIF9
PF00258, A0A7K5JV19
PF00258, A0A811NUG6
PF00258, A0A098VSA3
PF00258, A0AAT9URG5
PF00258, G7E2Y0
PF00258, A0A210PZ33
PF00258, A0A9P6SQY3
PF00258, A0A166NT13
PF00258, W3VGI8
PF00258, A0A7K7XQR7
PF00258, A0A4Y7NI32
PF00258, A0A3Q4AK65
PF00258, A0A194XWK7
PF00258, A0A7J8EFF8
PF00258, A0A7L3VZG0
PF00258, A0A6J1DH87
PF00258, A0A507R0M7
PF00258, A0A5M9JKL2
PF00258, A0A395IXF6
PF00258, A0A5N6K7N5
PF00258, A0A8A3PLI9
PF00258, A0A0W0FQX5
PF00258, V2WN84
PF00258, F6YXT2
PF00258, A0A4U1EPS2
PF00258, A0A3Q3J4N9
PF00258, A0A0D2NA21
PF00258, A9V987
PF00258, A0A4Q4TAP3
PF00258, A0A3N4LGN6
PF00258, A0A6A1VJE1
PF00258, A0A9P8D1W9
PF00258, A0A9P6F6Z2
PF00258, A0A8H7PCD2
PF00258, A0A9P6PNC7
PF00258, W9RQ06
PF00258, A0A8C6EGT2
PF00258, A0A7K5CL23
PF00258, A0A7S2W749
PF00258, A0A0C9N5G3
PF00258, S2K784
PF00258, A0A8H4F7C7
PF00258, A0A162Q9V2
PF00258, A0A8H7V159
PF00258, A0A8H7VA61
PF00258, A0AAN7I1Y5
PF00258, A0A371IF33
PF00258, A0AAW0MG44
PF00258, A0AAD4QMN2
PF00258, A0A5N3W6L2
PF00258, A0A5N3X9P8
PF00258, A0A9Q0EPZ9
PF00258, A0A6P7QYZ1
PF00258, F6XCS1
PF00258, A0A8C6HSP8
PF00258, A0A3Q8HZ68
PF00258, A0A8D7F234
PF00258, A0A4S8JRC7
PF00258, A0A9E7K3V3
PF00258, A0A9J7I9S2
PF00258, M3Y6X5
PF00258, A0AAD7E691
PF00258, A0AAD6TH75
PF00258, A0AAD6XFC6
PF00258, A0A8H6WIX9
PF00258, A0AAD2JYE5
PF00258, A0A8H6WHL7
PF00258, A0AAD7N1Q0
PF00258, A0AAD7ISK1
PF00258, A0AAD6YJ95
PF00258, A0AAD7GQH5
PF00258, A0A8H7DJY6
PF00258, A0A8H7CTW9
PF00258, A0A151X4A8
PF00258, A0A9W7ZU56
PF00258, Q4P3D8
PF00258, A0AAN7PJD7
PF00258, A0A7K9JF97
PF00258, A0AAW0JQA8
PF00258, S7P4M8
PF00258, L5M0M4
PF00258, G1PXQ0
PF00258, A0A7J7Y2J1
PF00258, A0A9P4J5M5
PF00258, A0A667ZD98
PF00258, A0A7L2SA94
PF00258, H9TN28
PF00258, A0A6A6Z7B7
PF00258, A0A6J8CV25
PF00258, A0A8S3U3B1
PF00258, A0A8B6F0F2
PF00258, A0A6B2G523
PF00258, A0A1E3PUG5
PF00258, A0A6A5CBB0
PF00258, D2VJZ3
PF00258, A0AA88KLP6
PF00258, A0A1Y2AMM0
PF00258, A0A8H3YEC8
PF00258, A0A8C6XV01
PF00258, A7WPE6
PF00258, W7U0J4
PF00258, A0A4D9DCX0
PF00258, A0A8C6RUL0
PF00258, A0A7M7TAC2
PF00258, G0VII1
PF00258, G0WHB7
PF00258, A0A6G5T7L0
PF00258, W2SWT0
PF00258, A0A822XQ17
PF00258, H8ZCN0
PF00258, A0A177ELM7
PF00258, I3EGE8
PF00258, A7T4X1
PF00258, A0A9Q0AL08
PF00258, A0A7S1R4I7
PF00258, A0A1Y2D609
PF00258, A0A9W9CL35
PF00258, A0A6J0C630
PF00258, A0A7L2RKL3
PF00258, A0A8C6WUQ2
PF00258, A0A6A6PSP3
PF00258, A0A3Q4HWX5
PF00258, A0A1U7LTP9
PF00258, A0A165PCW6
PF00258, A0A8M1MYE9
PF00258, A0A0P7BF63
PF00258, A0A341C0E6
PF00258, A0A7K4R531
PF00258, A1DK57
PF00258, F0VQX9
PF00258, A0A1A6GK17
PF00258, U6DT15
PF00258, A0AAD3STF2
PF00258, A0A8X6QMX5
PF00258, A0A6H5H089
PF00258, A0A7K7RU76
PF00258, A0A091T3A6
PF00258, F5HI28
PF00258, Q7SHF6
PF00258, A0AAJ0MU86
PF00258, F8N2U7
PF00258, A0AAE0J8W5
PF00258, A0A9P0HNY5
PF00258, A0A852I391
PF00258, A0A314L8C8
PF00258, A0A1U7YKF7
PF00258, A0A1S3XJ24
PF00258, A0A1S6KZV9
PF00258, A0A091VR86
PF00258, A5HVZ7
PF00258, A0A0N4YQ38
PF00258, A0A9K3Q5U0
PF00258, A0A167VE80
PF00258, A0A1J3K4W2
PF00258, A0A7S1FKJ8
PF00258, G1RS61
PF00258, A0A7S2VTX7
PF00258, R0KQL4
PF00258, A0A9P6GZR0
PF00258, A0A6J1UIC3
PF00258, I3NIN6
PF00258, A0A8C6NQ35
PF00258, A0A1A8DJR0
PF00258, A0A1A8GUH0
PF00258, A0A1A8JKL0
PF00258, A0A1A8PC47
PF00258, A0A1A8SGH8
PF00258, A0A7K7WHE5
PF00258, A0A851TCB5
PF00258, A0A7K7B7U8
PF00258, A0A7K6ZWP5
PF00258, A0A8C6ZRL3
PF00258, A0A7K6VRP0
PF00258, A0A7R9BFM1
PF00258, A0A6I9PAN6
PF00258, A0AAW0ETR3
PF00258, A0A811ZP78
PF00258, A0A7K8T7Q4
PF00258, A0A7L2GA25
PF00258, A0A7L1HN15
PF00258, A0A7L4CMK4
PF00258, A0A5K1HLG0
PF00258, A0A5J5BTG2
PF00258, A0A1L8E075
PF00258, A0A8E2DFZ0
PF00258, A0A7K8TTD9
PF00258, A0A7K9ML65
PF00258, B8Y644
PF00258, A0A288Z5I4
PF00258, E6Y368
PF00258, A0A288Z6B4
PF00258, L7X079
PF00258, A0A7S2MIB5
PF00258, A0A6P6DS06
PF00258, A0A0L8HMQ9
PF00258, A0A7E6EI16
PF00258, A0A2U3X3R4
PF00258, A0A6J0WUX9
PF00258, A0A7S4K005
PF00258, A0A348G635
PF00258, A0A7K9YNY5
PF00258, A0AAD9RZB0
PF00258, A0AAV6UF89
PF00258, A0A7L1E5A2
PF00258, A0A0B1ST39
PF00258, A0AAN6D4T9
PF00258, W1Q8S6
PF00258, A0A9P8T5M0
PF00258, A0A9P8TH11
PF00258, A0A0C3HDV9
PF00258, E4YKA2
PF00258, A0AAV1EAM3
PF00258, A0A8S0V950
PF00258, A0AAV9JR19
PF00258, A0A8H7ZVN5
PF00258, A0A238C219
PF00258, A0A182EJ92
PF00258, A0A2K6VKM5
PF00258, A0A8C7D837
PF00258, A0A8C7UDT2
PF00258, A0A8C8EQT3
PF00258, A0A7K5ZRH7
PF00258, A0A7J6BLH1
PF00258, A0A3L8DDY7
PF00258, A0AAV7J8G1
PF00258, A0A0L7LVL6
PF00258, A0A6A7A793
PF00258, A0A2C5XFX9
PF00258, A0A8H4Q609
PF00258, A0A2C5ZGA2
PF00258, A0A367LGV3
PF00258, A0A8H4Q000
PF00258, T5AJV3
PF00258, A0A2A9PPC6
PF00258, V8P1I9
PF00258, Q8H0D6
PF00258, S3CQL2
PF00258, A0A091VBN4
PF00258, A0A4S2M971
PF00258, A0A1S8WIB1
PF00258, A0A7R9MCN2
PF00258, A0A7C9E9B2
PF00258, A0AAV9VQA3
PF00258, A0AAV9UCU8
PF00258, A0AAV9XU83
PF00258, A0AAN8RFY0
PF00258, A0A7C8Q5V1
PF00258, A0A1D2N5F8
PF00258, A0A0B2UMP9
PF00258, A0A7K6JZ28
PF00258, A0A668U3X2
PF00258, A7DZ98
PF00258, A0A669F7Y7
PF00258, A0A7L3NG46
PF00258, A0A7K6DCP7
PF00258, A0A7L1PH51
PF00258, A0A293MQS4
PF00258, A0A1Z5KYS4
PF00258, A0A2R5L485
PF00258, A0AAU8JND8
PF00258, A0A6I8PEF9
PF00258, A0A7K8H3W1
PF00258, A0A8B7B101
PF00258, A0A0T6BEL7
PF00258, Q9GLN0
PF00258, A0A0D3GXW4
PF00258, J3MRI3
PF00258, I1PFZ5
PF00258, A0A0D9ZRR5
PF00258, A0A0E0EBR7
PF00258, A0A6G1ED71
PF00258, A0A0E0GUF9
PF00258, A0A0E0KW35
PF00258, A0A0E0P1Y2
PF00258, A2XMF8
PF00258, B9FCW2
PF00258, A0A3S2MGI0
PF00258, H2LUA3
PF00258, A0A3B3CUP2
PF00258, A0A8C8DG80
PF00258, A0A8S1J5W4
PF00258, A4S197
PF00258, A0A7S0WCK5
PF00258, A0A7R9T2H3
PF00258, A0A1Y5I6N1
PF00258, A0A3G2BAG7
PF00258, H0XFM4
PF00258, A0A8C8ECP4
PF00258, A0AAD4TYW8
PF00258, W5PYL0
PF00258, T2HPQ4
PF00258, A0A8S4N482
PF00258, A0A7S4LPA5
PF00258, A0A7L0ZX72
PF00258, A0A7K9PLY9
PF00258, A0A7K5D754
PF00258, A0A1E4U3E0
PF00258, A0ABD2FGW7
PF00258, A0A7S3G6A1
PF00258, A0A7L4FNX2
PF00258, A0A2R9CLM9
PF00258, A0A2J8PX61
PF00258, A0A409VES9
PF00258, A0A7E4V9Q0
PF00258, A0A914P6A2
PF00258, A0A914Z4U8
PF00258, V5LFT5
PF00258, A0A7L2WJJ8
PF00258, A0A5N5M6Q5
PF00258, A0A2T8IK81
PF00258, A0A2T7C0F1
PF00258, A0A3L6R2V6
PF00258, A0A8T0QX21
PF00258, A0A0U1WZ99
PF00258, A0A224XL98
PF00258, A0A069DZD2
PF00258, A0A8C9DAU7
PF00258, A0A9W2VWI5
PF00258, A0A8C9JKM9
PF00258, A0A6P9BHH4
PF00258, D2J3S3
PF00258, A0A7K6MA53
PF00258, A0AAD4XDX4
PF00258, A0AA41SFW4
PF00258, A0A4Y7K5C2
PF00258, A0A0N1IJL4
PF00258, A0A194PK34
PF00258, A0AAD9CW48
PF00258, A0A2I3MQC2
PF00258, A0AAN6PMF5
PF00258, A0A1D2J773
PF00258, A0A0A0HS83
PF00258, A0A0A2V600
PF00258, A0A9N9GF23
PF00258, A0A9N8WAM4
PF00258, A0A8J4SK47
PF00258, A0A8S9YPS6
PF00258, A0A8T0D3U6
PF00258, A0AAD9JIR1
PF00258, A0AAW0E7B3
PF00258, A0A6P7KH90
PF00258, A0A8S1YGZ5
PF00258, A0A8S1YQ38
PF00258, A0A8S1Q5I0
PF00258, A0A8S1R6S3
PF00258, A0EE80
PF00258, A0A2H9TLW4
PF00258, A0A7S4PNZ2
PF00258, A0A3B3RQU5
PF00258, A0A7D9MAV2
PF00258, A0A9P6GSW0
PF00258, A0A177CRI4
PF00258, A0A8K0W015
PF00258, S4NUZ8
PF00258, A0A8S4RD32
PF00258, A0A915BXU0
PF00258, A0A9P1GUU5
PF00258, A0A0B7NW80
PF00258, A0A2P5DY49
PF00258, A0A2L2Z6B7
PF00258, A0A0N4Z5B8
PF00258, A0AAN6U5V7
PF00258, A0AAN6PXK8
PF00258, A0A7L3IN85
PF00258, A0AAD5WKJ7
PF00258, A0A8S3XHH0
PF00258, A0AAV1LKT9
PF00258, A0AAN9TP17
PF00258, A0AAQ3X9W2
PF00258, A0A9Q8UR70
PF00258, A0A852DEC6
PF00258, A0A140B165
PF00258, A0A1V4L116
PF00258, A0AAN8JWN7
PF00258, A0A9P4SA38
PF00258, A0A914BQR4
PF00258, B1X4E1
PF00258, A0A2H4ZPA9
PF00258, A0A385I069
PF00258, A0A8C9FYZ7
PF00258, A0A0C9TZN6
PF00258, A0A0D0C6M2
PF00258, E0VTT3
PF00258, A0A7K6P0N0
PF00258, A0A8J2WZB9
PF00258, A0A7L3C2M3
PF00258, A0A091SDK4
PF00258, A0AAD1RYG8
PF00258, K7G8D9
PF00258, A0A6H0XW98
PF00258, A0A8C8VLY2
PF00258, I2CZU9
PF00258, D2YYH5
PF00258, G8XR43
PF00258, A0A3R7PPE6
PF00258, A0A851NM34
PF00258, A0A1L9SVI2
PF00258, A0A9W9ER95
PF00258, A0A9W9GCZ0
PF00258, A0A1V6QE16
PF00258, A0A9W9EXA6
PF00258, A0A1F5LFS3
PF00258, A0A9W9UA85
PF00258, A0A9W9KWV0
PF00258, A0A0F7VFQ7
PF00258, A0A9W9UR64
PF00258, A0A0G4PMW7
PF00258, A0A9W9LQ31
PF00258, A0AAD6I6E0
PF00258, A0A9W9HU60
PF00258, A0A9W9V7D5
PF00258, A0A9W9MG96
PF00258, A0A9W9T886
PF00258, A0A9W9P6Y7
PF00258, A0A167U6L6
PF00258, A0A9W9NIT3
PF00258, A0A9W9PBJ7
PF00258, A0A9W9UVU9
PF00258, A0A1V6V0U7
PF00258, A0A9W9VS92
PF00258, A0A9P5GCI1
PF00258, A0AAD6C2S7
PF00258, A0A1V6PHI0
PF00258, A0A9W9WFL3
PF00258, A0A9X0BU66
PF00258, A0A7T6XJJ6
PF00258, K9GA12
PF00258, K9G4M4
PF00258, A0A9W4KRW7
PF00258, A0A0A2JEF8
PF00258, A0A9W9Y741
PF00258, A0A1V6TMR1
PF00258, A0A101MG99
PF00258, A0AAD6CP96
PF00258, A0AAD6GSN2
PF00258, A0AAD6E9I0
PF00258, A0A0A2LFU1
PF00258, A0AAD6MYC1
PF00258, A0A1V6Y7T7
PF00258, A0A0M8P1G2
PF00258, A0A9W4H985
PF00258, S8BCN6
PF00258, A0A135LCN8
PF00258, A0A1V6NCP6
PF00258, W6QRF5
PF00258, B6HMT8
PF00258, A0A9W4P154
PF00258, A0A1V6QT79
PF00258, A0A1V6TMD1
PF00258, A0A1Q5UKB3
PF00258, A0AAI9TEU6
PF00258, A0A8J8W1R0
PF00258, A0A1V6RTL8
PF00258, A0ABD3U8V5
PF00258, A0A484C6D9
PF00258, A0A6A5EA33
PF00258, A0A7S1KU93
PF00258, A0A9W4U3R2
PF00258, A0A2V1DPV6
PF00258, D4N1M4
PF00258, A0AAD4JMH7
PF00258, A0A3B4B5T3
PF00258, A0A7J6N2W3
PF00258, C5LRV2
PF00258, A0A7J6UJ64
PF00258, A0A8C9CU01
PF00258, A0AAU9KYF2
PF00258, A0AAV0TP85
PF00258, A0A3R7WSJ6
PF00258, A0AAV0TH28
PF00258, A0AAV1TA15
PF00258, W3X6Q0
PF00258, A0AAE1FZP5
PF00258, A0AAE1UJF8
PF00258, A0A8H6A993
PF00258, S4RSF0
PF00258, O48938
PF00258, B3RFK3
PF00258, A0A852FT59
PF00258, A0A9P0GSB7
PF00258, R8BQF4
PF00258, A0A7S0EZP9
PF00258, A0A8J9X989
PF00258, B7GCM3
PF00258, A0A7S1XTP7
PF00258, A0A0G2EZ40
PF00258, Q0UIA4
PF00258, B6ZIU6
PF00258, A0A091UHP5
PF00258, A0A7L4BIX4
PF00258, A0A0F7ST72
PF00258, A0A7L1U255
PF00258, A0AAV0BG56
PF00258, A0A093R0P9
PF00258, A0A6F9DM10
PF00258, K5WD40
PF00258, F1T2K0
PF00258, G5EJY5
PF00258, B5LZ07
PF00258, A0AAD5PGB2
PF00258, A0A8T0K3S5
PF00258, A0AAN9NTQ3
PF00258, V7CRL3
PF00258, A0A669PPL1
PF00258, A0A4S4KQP4
PF00258, A0A7K7DKA9
PF00258, A0AAJ0BS78
PF00258, A0A1L7XQM7
PF00258, A0A0D2FW13
PF00258, A0A0K0LJ72
PF00258, A0A411KVE0
PF00258, A0A0B8RX23
PF00258, A0A0C3SDT0
PF00258, A0A6B2EDY0
PF00258, A0A1B0DDL2
PF00258, A0A8C9BHG2
PF00258, A0AAV0A7T8
PF00258, A0A091U160
PF00258, A0A8B8J3L4
PF00258, A0A9P5YSI6
PF00258, A0AAD9W9B9
PF00258, A0A1Y1N4Q1
PF00258, A0AAN9GRJ2
PF00258, A0A0R5P8F4
PF00258, A0A9Q0Y1H6
PF00258, A0A0R5PC58
PF00258, A0A830BRT8
PF00258, A0A167LVR7
PF00258, A0AAD9I4J1
PF00258, A0A834EQW8
PF00258, A0A9P0GP01
PF00258, Q968Y5
PF00258, A0A2K1KMY3
PF00258, A0A455BIN1
PF00258, A0AAD5UV75
PF00258, A0AAD5T8D3
PF00258, A0A8J5J8S3
PF00258, A0A8T1X5F8
PF00258, A0A8T1UJN6
PF00258, A0AAD9LIN7
PF00258, A0A6G0RPC4
PF00258, A0A9W7CL89
PF00258, A0A8S9VB68
PF00258, D0P304
PF00258, A0A922AS98
PF00258, A0A8J4WN81
PF00258, A0A9W6WQQ2
PF00258, A0A225VY21
PF00258, W2JVP3
PF00258, W2QDM4
PF00258, W2XUR2
PF00258, W3A498
PF00258, V9FYX5
PF00258, A0A081B2S9
PF00258, A0ABD3FM00
PF00258, A0A2P4YBW9
PF00258, A0A8T1VX62
PF00258, H3HCW1
PF00258, A0A6A4EWR4
PF00258, G4ZGW0
PF00258, A0A2I6PCQ6
PF00258, A0A850WIG1
PF00258, A0A851BVV7
PF00258, A0AAN6I6R4
PF00258, A0A9P6WIV7
PF00258, A0A4V4NG57
PF00258, A0AAV5RAG1
PF00258, A0A2U9R6H6
PF00258, A0A1Q2YDR9
PF00258, A0A1E3NN15
PF00258, G8Y946
PF00258, A0A7S3UFK1
PF00258, G9JVI0
PF00258, A0A6A7C500
PF00258, A0A9P0TN79
PF00258, A0A821RA78
PF00258, A0A2S0RQR2
PF00258, A0A8C9GK39
PF00258, A0A0C3BJJ7
PF00258, A0AA89BJT8
PF00258, A0A7R9U2N2
PF00258, A0A077D6G6
PF00258, A0A7T9QPQ1
PF00258, A0A7J7YYL5
PF00258, A0A7R5KYD1
PF00258, A0A7L0J131
PF00258, A0A4P9Y0W9
PF00258, A0A1Y1VEX1
PF00258, A0A0C9YKN1
PF00258, A0A0C3JFA6
PF00258, A0A9D5B6S4
PF00258, A0A851FVN4
PF00258, A0A7G7LKC4
PF00258, A0AAV4DNF5
PF00258, A0A2P6NMT6
PF00258, A0A3P3Y1S4
PF00258, A0A113SIQ3
PF00258, A0A509APE3
PF00258, A0A1D3LLF0
PF00258, A0A4V0KA66
PF00258, A0A1B1DXD3
PF00258, K6ULW8
PF00258, Q8IKX3
PF00258, W7EUC3
PF00258, A0A024XCI1
PF00258, A0A0L7M4T3
PF00258, A0A0L7KEB8
PF00258, W7JZ69
PF00258, W4J0P7
PF00258, A0A024VLD2
PF00258, A0A0L1IE16
PF00258, A0A024WIU8
PF00258, W4I9P6
PF00258, A0A0L0CYI6
PF00258, W7FHK8
PF00258, A0A024WBH2
PF00258, W7JQA8
PF00258, A0A024UZ30
PF00258, A0A0D9QIT3
PF00258, A0A151LAV9
PF00258, A0A1J1GQY0
PF00258, A0A1Y1JRE4
PF00258, W7AKT0
PF00258, A0A1Y3DW33
PF00258, A0A5K1UTJ5
PF00258, A0A1D3SQV7
PF00258, A0A1D3U8F0
PF00258, A0A1A8WI97
PF00258, A0A1A8YR38
PF00258, A0A2P9DQ34
PF00258, A0A1J1H9E7
PF00258, A0A6V7TBR5
PF00258, A0A6V7SLZ8
PF00258, A0A6V7SQK8
PF00258, W7AI02
PF00258, A0A449BY31
PF00258, A0A8S4H9M6
PF00258, A0A0J9ST96
PF00258, A5K3Q5
PF00258, A0A0J9SAB1
PF00258, A0A0J9T9D8
PF00258, A0A0J9TQ50
PF00258, A0A077Y8V2
PF00258, V7PPE7
PF00258, Q7RE87
PF00258, A0A0P1APL1
PF00258, A0AAP0BLP1
PF00258, A0A7K5VW33
PF00258, A0A4D9F0L7
PF00258, A0A8K0TE55
PF00258, A0A9P9A9Y3
PF00258, A0A1U9ZCB6
PF00258, C0Z3X6
PF00258, A0A914WD81
PF00258, A0A6A7BJF4
PF00258, A0A9W6EZZ9
PF00258, A0A6G1KI38
PF00258, A0AAV7VTS7
PF00258, A0A9N7VX56
PF00258, A0AA38VLF3
PF00258, A0A9P5ZQD1
PF00258, A0A8H7DSQ7
PF00258, A0A067NWA2
PF00258, A0PFJ4
PF00258, A0A7L0Z4T8
PF00258, A0A8S4ES59
PF00258, A0A7L3DJ13
PF00258, A0A0W4ZJB7
PF00258, L0P7P1
PF00258, A0A0W4ZVU1
PF00258, M7P519
PF00258, A0A899FZB8
PF00258, A0A179FIS1
PF00258, A0A3M6TY43
PF00258, A0AAU9W2J9
PF00258, A0AA35PR31
PF00258, A0A670K8K5
PF00258, A0A7L4GVK0
PF00258, A0A094KR78
PF00258, A0A9P5SU26
PF00258, A0A086TJW2
PF00258, A0A7L0TBW5
PF00258, A0AAV9GK70
PF00258, B2B2Y5
PF00258, A0AAE0X9R6
PF00258, A0AAN6X3I8
PF00258, A0AAE0P0Y0
PF00258, A0AAN7BUZ1
PF00258, A0A7K7R3W9
PF00258, A0A087X6J2
PF00258, A0A3B3TR64
PF00258, A0A3B3X050
PF00258, A0A3P9PGF4
PF00258, A0A6J0UIV6
PF00258, A0A6I9WSK1
PF00258, A0AAD6B138
PF00258, A0A813LFC0
PF00258, A0A7K5EM96
PF00258, A0A9P4QB46
PF00258, A0A9J6CP56
PF00258, A0AAN8PJP2
PF00258, A0A9P4V776
PF00258, A0A5C3PMI7
PF00258, A0A8X8BLP6
PF00258, A0A8J4Q156
PF00258, A0A2B7Y8B5
PF00258, A0A7S0VL54
PF00258, A0A2T7NH46
PF00258, A0A7L4J4R7
PF00258, A0A7L2TWG6
PF00258, A0A2J8XU44
PF00258, A0AAW1M496
PF00258, A0A4U5PPQ9
PF00258, A0AAD6W3A2
PF00258, A0A6M2E974
PF00258, A0A8T2ZFW9
PF00258, A0AAJ6US09
PF00258, A0A8X8DAY4
PF00258, A0A3N7FYT4
PF00258, Q9AU08
PF00258, A0A836L6S5
PF00258, A0A1X6NIT7
PF00258, A0A5J4Z3P3
PF00258, A0A5B7DZP8
PF00258, A0A1X6MKL3
PF00258, A0AAE0W8D6
PF00258, A0A507DTX3
PF00258, A0A7R9U1F6
PF00258, A0A7S3C0B6
PF00258, A0AAV5SV97
PF00258, A0AAV5VG84
PF00258, A0AAN4ZMA4
PF00258, A0A2A6C975
PF00258, A0A7S0GN34
PF00258, A0A7K5FG61
PF00258, A0A8C8ZBS3
PF00258, A0A7K6XW74
PF00258, A0A2K6FUP3
PF00258, A0A2K8DPD4
PF00258, A0A9Q0QPR9
PF00258, A0A1Y2F6K2
PF00258, A0A3S5CKD1
PF00258, A0AAD9IPF0
PF00258, A0A7L3A807
PF00258, A0A7K5QY29
PF00258, A0A6J5WT13
PF00258, A0A6P5TGK1
PF00258, A0A4Y1QVN5
PF00258, A0A251Q4P1
PF00258, A0A140B4A9
PF00258, A0A314YHQ8
PF00258, A0AB34J0J0
PF00258, A0A084G8S9
PF00258, A0A7R9ZFL5
PF00258, A0A7S4EHZ1
PF00258, A0A7S0UF86
PF00258, G9FYY8
PF00258, A0A448Z695
PF00258, A0A836FGG5
PF00258, A0A139HLP7
PF00258, M3API1
PF00258, A0A8H6VJU1
PF00258, A0A139IPW3
PF00258, A0A0V0R6Z5
PF00258, A0A177AFY6
PF00258, L8GAZ4
PF00258, A0A1B8GKP7
PF00258, A0A0R0LYL1
PF00258, A0A9Q0RRW5
PF00258, A0A1Y2EFR5
PF00258, A0A316U3Q9
PF00258, A0A670ZVK6
PF00258, A0AAN6NYQ7
PF00258, A0AAN6LR48
PF00258, Q40916
PF00258, A0A6A6VWU7
PF00258, A0A5C3FV32
PF00258, M9LYZ5
PF00258, A0A5C3F0H1
PF00258, A0A061HBR3
PF00258, R9P804
PF00258, A0A8H5FAF4
PF00258, A0A8H7XTQ8
PF00258, A0A409WZI8
PF00258, A0A7K9CGN9
PF00258, A0A7K9Y0I5
PF00258, A0AAN9XUK5
PF00258, A0A9P0GDA0
PF00258, A0A7K5YEF5
PF00258, A0AAW3DKI1
PF00258, L5KCB8
PF00258, A0A6P6CY84
PF00258, A0A5C3QNW3
PF00258, A0A852MJG0
PF00258, A0A7K6CLE2
PF00258, A0A7K8MSN2
PF00258, A0A2N5W4Y8
PF00258, A0A5B0N1E3
PF00258, E3KDF1
PF00258, A0A0L6V1K6
PF00258, A0A2S4VM45
PF00258, A0A0L0UVD2
PF00258, A0A180GK89
PF00258, A0A6P6HX73
PF00258, A0A3B4ETS2
PF00258, A0A2I0L6Q4
PF00258, A0AB34FXF3
PF00258, A0A2U3DUF2
PF00258, A0A9Q8VEL4
PF00258, A0A830HM53
PF00258, A0A7L2P9A7
PF00258, A0A060SIF4
PF00258, A0AAR2M0J9
PF00258, A0A093P2P2
PF00258, A0AA40H4V7
PF00258, A0A7S0WLP9
PF00258, A0A3M7MDN2
PF00258, A0A6S6WA64
PF00258, E3S8Q0
PF00258, A0A834SC12
PF00258, B2WGI7
PF00258, A0A6P8BKC8
PF00258, A0A4P7NC51
PF00258, G4N597
PF00258, L7JKB8
PF00258, A0AA97P165
PF00258, A0AAN7VB68
PF00258, A0A7S0FDM8
PF00258, U4KXY6
PF00258, A0A286UPZ1
PF00258, A0A5N5HTD3
PF00258, A0AAD5Q714
PF00258, A0A8K1C6H9
PF00258, A0A9F2WH52
PF00258, A0AAV3A8I7
PF00258, A0A7N2MX89
PF00258, A0AAN7G5U8
PF00258, A0AAW0LJ98
PF00258, A0AAD7LXC9
PF00258, A0A7L2FIQ4
PF00258, A0A9N9NTV4
PF00258, A0AA43TSC6
PF00258, A0A1D1VWF5
PF00258, A0A852CRB2
PF00258, A0A2D3VI54
PF00258, D0VP29
PF00258, A0ABD0YYA9
PF00258, A0A897Q6U5
PF00258, A0A9W3BQN1
PF00258, A0A2V0P8A7
PF00258, A0A0F4YMB2
PF00258, A0ABZ3NNP0
PF00258, A0AAE1C453
PF00258, A0A7K4XPM9
PF00258, X6MM93
PF00258, A0A140JTI7
PF00258, A0A7K9LEA5
PF00258, A0A7L2MTK4
PF00258, A0A7K8CAT2
PF00258, A0A8K0H5M3
PF00258, A0AAV8ZT32
PF00258, A0A973YIN3
PF00258, G4V397
PF00258, A0A0D2IXV5
PF00258, A0A7J7VR75
PF00258, A0A2K6LB07
PF00258, A0A2K6NG02
PF00258, A0A7L1NVG6
PF00258, A0A131YK04
PF00258, A0A9J6F347
PF00258, L7MG88
PF00258, A0A9D4SZ28
PF00258, A0A224Z3Y9
PF00258, A0A7K9WAB7
PF00258, A0A7S2SA59
PF00258, A0A1Y2D173
PF00258, A0A8H8SUG5
PF00258, A0A074SAH4
PF00258, X8J216
PF00258, A0A9P4INR2
PF00258, A0A2Z6RQE1
PF00258, A0A2N1NQ79
PF00258, U9U9M0
PF00258, A0A015LQQ6
PF00258, A0AAD5X377
PF00258, A0A2P2MDR5
PF00258, A0A1J8QZ43
PF00258, A0A1B7MIP5
PF00258, A0A367JNC3
PF00258, A0A9P7CRV9
PF00258, I1CVQ4
PF00258, A0A1X0SFF2
PF00258, A0A2G4T7V8
PF00258, A0A1X0RC48
PF00258, A0A9P6WXH2
PF00258, Q9HFV4
PF00258, A0A8B8MXK1
PF00258, A0A7K8S9Z2
PF00258, A0A0P4VT25
PF00258, T1HKS3
PF00258, A0A9P5U9W7
PF00258, A0AAV6LHA8
PF00258, A0A834LJA2
PF00258, A0A6A4LKJ4
PF00258, A0A4Y9Y8B5
PF00258, A0A8H7U1I4
PF00258, A0AAV8UQB6
PF00258, A0A5C5G5B6
PF00258, A0A194SAR7
PF00258, A0A9P7BA09
PF00258, A0AAV5GWP8
PF00258, A0A2S5B908
PF00258, A0A0K3CTJ5
PF00258, M7XIZ7
PF00258, A0A172R1M8
PF00258, A0A834M6E8
PF00258, A0A9Q0HY73
PF00258, A0AAV8DW67
PF00258, A0AAD5ZZX9
PF00258, A0A1E1JSL8
PF00258, A0A1E1KPR7
PF00258, A0A1E1MSQ8
PF00258, A0A7K6RWU4
PF00258, A0AAW1DKQ0
PF00258, A0AAN6XZB8
PF00258, A0ABD1ZEY4
PF00258, A0ABD3IFT8
PF00258, B9THS7
PF00258, A0A4Y7QGZ0
PF00258, A0AAD9P9C6
PF00258, A0A7L3TE58
PF00258, A0A0R3T7U4
PF00258, A0A915ISR7
PF00258, A0AAD7G2Q2
PF00258, A0A2P6RML3
PF00258, A0A1W2TS89
PF00258, A0A6G7GFL4
PF00258, A0A7L0DI54
PF00258, A0A8S3H5X5
PF00258, A0A821Z5I9
PF00258, A0A820FKY3
PF00258, A0A7J8F2V6
PF00258, A0A4V1IYS8
PF00258, A0AAV5K8H5
PF00258, A0AAW1YE79
PF00258, A0A9P5T9A9
PF00258, A0A7L1KGW6
PF00258, A0A9P4I029
PF00258, J8Q7F9
PF00258, E9P9H6
PF00258, Q12181
PF00258, B5VEV2
PF00258, N1P814
PF00258, C7GQL7
PF00258, G2WA22
PF00258, C8Z489
PF00258, B3LU76
PF00258, A6ZTI6
PF00258, H0GRV4
PF00258, J8TQZ5
PF00258, A0AA35IYS5
PF00258, A0A8B8UMF3
PF00258, A0A6C1E435
PF00258, A0AA35NQ01
PF00258, A0A376B486
PF00258, A0AAV5QHE1
PF00258, A0A7L2HJ52
PF00258, A0A2K6SWN2
PF00258, A0A0E9NJ67
PF00258, A0A427YR67
PF00258, A0A7K8YLL6
PF00258, A0A672JJC8
PF00258, A0A4U0TVI3
PF00258, A0A5N5NRK3
PF00258, A0A835TI80
PF00258, A0A9Q0ZLM8
PF00258, A0A9Q0V2P7
PF00258, A0AAD6NW80
PF00258, A0A6N2LDX9
PF00258, A0A1S3QB44
PF00258, A0A674ERT1
PF00258, F2US29
PF00258, A0A8D0CDK3
PF00258, A0A8U0TUP5
PF00258, A0ABD1IGB2
PF00258, S4URU2
PF00258, A0A8X8XD92
PF00258, A0A8D0D549
PF00258, A0A9Q5I329
PF00258, V5RH79
PF00258, A0A6J3GJN6
PF00258, A0A7K7T6H5
PF00258, A0AAW1ML91
PF00258, T0PYA1
PF00258, A0A067CN78
PF00258, A0A7N4PMF1
PF00258, A0A834RE76
PF00258, A0AA39GGY5
PF00258, A0AAV9NWH7
PF00258, A0A4Y7NMM1
PF00258, A0A4Y5QZP3
PF00258, A0A7T3CIL7
PF00258, A0A9P8AK46
PF00258, A3LT99
PF00258, A0A499SZA0
PF00258, A0A3P7D0Q3
PF00258, A0A430Q904
PF00258, A0A183KM16
PF00258, A0A922LGL6
PF00258, C1L3S0
PF00258, G4LUL4
PF00258, A0AA84ZRW7
PF00258, A0AA85BE44
PF00258, A0AAE1ZER4
PF00258, A0AA85ETX4
PF00258, A0A2S2NBX2
PF00258, A0A550C1X1
PF00258, D8Q3J6
PF00258, A0A0H2SCW6
PF00258, S9VV98
PF00258, B6K8E8
PF00258, S9QW59
PF00258, A0AAF0AWY8
PF00258, O59761
PF00258, A0AA40EVX9
PF00258, C8YMC8
PF00258, A0AA41T9U7
PF00258, A0A8D2JSP8
PF00258, A0A0C2ZWA3
PF00258, A0A8D0CKM6
PF00258, W9CSV7
PF00258, A0A9X0AD11
PF00258, A7F276
PF00258, A0A8H2ZTK6
PF00258, A0A7K8WRA2
PF00258, A0AAV1PL28
PF00258, U5N352
PF00258, A0A8D3CE60
PF00258, A0A7L4HWQ3
PF00258, Q2KM85
PF00258, A0A401Q211
PF00258, A0A0P4WEL1
PF00258, A0AAW0UPJ4
PF00258, A0A3E2H0F6
PF00258, A0A7L1Z4Y4
PF00258, D8RYS1
PF00258, A0A9N8H7P2
PF00258, A0A7L2IRF0
PF00258, A0A834TFA4
PF00258, Q2YHG0
PF00258, A0A9Q9B388
PF00258, G4TIZ0
PF00258, A0A0C2WJN1
PF00258, A0A7L1D494
PF00258, A0A8C9MNE4
PF00258, A0A3B4TAB8
PF00258, A0A3B4X3H4
PF00258, F8QHK9
PF00258, F8P3N1
PF00258, A0AAE1YPY2
PF00258, A0AAE1WG67
PF00258, A0AAW2N4A8
PF00258, A0AAW2RTJ0
PF00258, A0A6I9U9N9
PF00258, A0AAW2VY32
PF00258, A0AAW2SKH8
PF00258, A0A915Q739
PF00258, K3ZBW1
PF00258, A0A4U6VJN7
PF00258, A0A9P4HDM9
PF00258, A0A7L0QYX3
PF00258, A0A7S1VJU4
PF00258, A0AAD5FHJ9
PF00258, A0A8T0AJQ6
PF00258, A0A4Y7NND0
PF00258, A0ABD3V748
PF00258, A0A671S3G2
PF00258, A0A672LQW2
PF00258, A0A673HHX6
PF00258, A0A8K1WCA4
PF00258, A0A7K4UKT9
PF00258, A0A8B8GLP8
PF00258, A0A386TMJ5
PF00258, A0A164VQ92
PF00258, A0A166FHT2
PF00258, A0A6J2X4R5
PF00258, A0A6B9HAJ6
PF00258, A0A7L1VNP4
PF00258, A0AAD9DED9
PF00258, A0A7K8R0H6
PF00258, A0A2U1J2G7
PF00258, A0A1R1YQX0
PF00258, A0A2T9ZD44
PF00258, A0A1R0GRE1
PF00258, A0A2T9Y7A7
PF00258, A0A7L1JDI7
PF00258, A0A183J7R8
PF00258, A0A3N2Q1I9
PF00258, X2F552
PF00258, A0AAN8YLD3
PF00258, A0A0V0INH0
PF00258, A0A6N2CIB3
PF00258, A0A9J6AG60
PF00258, A0A3Q7H9M0
PF00258, A0AAV9LBQ6
PF00258, A0ABD2VD74
PF00258, M1AYF8
PF00258, A0AAF0UB12
PF00258, A0AAV6T443
PF00258, A0A0E3D9X6
PF00258, A0AAE0UDF9
PF00258, A0A8S9A434
PF00258, F7W2F2
PF00258, A0A921UXW0
PF00258, A0A484GT78
PF00258, A0A401H4C7
PF00258, A0A671XEQ2
PF00258, G3AU91
PF00258, A0A8C9QLW5
PF00258, A0A2K1QHR4
PF00258, A0A673AYD7
PF00258, A0A0C9V7W8
PF00258, A0A0L0G6Y3
PF00258, A0A5J5F061
PF00258, N1QDA2
PF00258, A0A9P7FV22
PF00258, A0A8J4MTQ6
PF00258, A0A8D0L3Z9
PF00258, A0AA86T1K4
PF00258, A0A9R0JAH9
PF00258, A0A7I8J5Y8
PF00258, Q86QX8
PF00258, V6M5G8
PF00258, A0A7L0BY38
PF00258, A0A0L0H847
PF00258, A0A922MLK4
PF00258, A0A9R0E4U2
PF00258, J7FJF0
PF00258, A0A9J7E1L9
PF00258, A0A0H5R9C0
PF00258, A0A0D6EMM6
PF00258, A0A4U7KQ46
PF00258, E7A1G0
PF00258, A0A2N8UEC1
PF00258, A0A127ZI96
PF00258, A0A6A6VL39
PF00258, A0A0C2ESL8
PF00258, U7Q127
PF00258, A0A0F2MI88
PF00258, S7W595
PF00258, A0A7S3GQ26
PF00258, Q9I9M2
PF00258, A0A084B1T4
PF00258, A0A084QV26
PF00258, A0A8K0SWG3
PF00258, A0AAD4EYT2
PF00258, A0AAN6MPI2
PF00258, A0AAV5RPT6
PF00258, A5Y0M3
PF00258, A0A7K6VYU0
PF00258, A0A4R0S2S8
PF00258, A0A3B5ALR3
PF00258, A0A087UP31
PF00258, A0A4U8UMD3
PF00258, A0A1I8AWI7
PF00258, A0AA39I5C3
PF00258, A0A364NF11
PF00258, Q9PTJ0
PF00258, A0A1R2BIP8
PF00258, A0AAP0ED38
PF00258, A0AAP0I5J4
PF00258, A0AAP0EEC5
PF00258, A0ABD3NVM5
PF00258, A0A7K9FK25
PF00258, R7S050
PF00258, A0A7K9SAN8
PF00258, Q2I6J8
PF00258, A0A2G8KNI7
PF00258, A0A1I8NTV0
PF00258, E1XU24
PF00258, A0A5A7QS33
PF00258, A0A9N7RN90
PF00258, T1JG33
PF00258, S9V782
PF00258, A0A672V3V1
PF00258, A0A8D0EF38
PF00258, A0A7S3XCB9
PF00258, A0A7M7PBV8
PF00258, A0A0N5BSL0
PF00258, A0A090MZ30
PF00258, A0AAF5I2F3
PF00258, A0A0K0FYV0
PF00258, A0A3P7J8H9
PF00258, A0A7K8FL15
PF00258, A0A093I4X6
PF00258, A0A192ZJF7
PF00258, A0A077ZSQ2
PF00258, A0A2B4RLE7
PF00258, A0A167G019
PF00258, A0A1E4SR10
PF00258, A0A9P7JWM7
PF00258, A0AAD4EBX2
PF00258, A0A0C9ZX08
PF00258, A0A9P7CWN6
PF00258, A0A9P7DKF3
PF00258, A0A9P7JDY3
PF00258, A0A851ARP8
PF00258, A0A1L0C281
PF00258, A0A673UUI1
PF00258, A0A8D2C9G9
PF00258, A0A7K7EG75
PF00258, A0A7L1FQL2
PF00258, A0A7L0LCS8
PF00258, A0AAW1PIT6
PF00258, A0A1Q9F3K8
PF00258, A0A812SNX7
PF00258, A0A813BN18
PF00258, A0A812X6E4
PF00258, A0A0N7ANB7
PF00258, A0A9Q1G719
PF00258, A0A1X2HW66
PF00258, A0A4P9Z0L4
PF00258, A0A507DSF8
PF00258, A0A507CEX3
PF00258, A0A0N5AMW7
PF00258, A0A7L3B6C8
PF00258, A0A0K8TSW9
PF00258, A0A7K4WJV3
PF00258, A0AA88IM13
PF00258, A0A0R3VXH9
PF00258, A0A674HG86
PF00258, A0AAD8P062
PF00258, A0A4Z2BVE7
PF00258, A0A5C6NUG0
PF00258, Q8JI76
PF00258, H2TGW3
PF00258, A0A364L099
PF00258, A0A225AMQ1
PF00258, A0A0U1M1R9
PF00258, B6QNL2
PF00258, A0A093XD99
PF00258, A0A6V8HBS5
PF00258, A0AAD4PS33
PF00258, A0A7H8R2T9
PF00258, B8MRE7
PF00258, R9WTT3
PF00258, R4X6J0
PF00258, Q1KUY9
PF00258, A0A093CAS0
PF00258, A0AA38F4G2
PF00258, Q5J0E4
PF00258, A0A2G9V1E2
PF00258, A0A6J1QPU2
PF00258, A0A4V3S7Q7
PF00258, A0A8J6HKC2
PF00258, A0A9W7VYI6
PF00258, A0A6G1L2Y8
PF00258, A0A3N4LG58
PF00258, A0A674JUY8
PF00258, A0A317Y1U0
PF00258, A0A2J8A783
PF00258, A0A834ZLL5
PF00258, A0A383WGI5
PF00258, A0AAW1A7Y1
PF00258, I7M6V6
PF00258, A0A0K0PSZ5
PF00258, T1KG33
PF00258, Q4T5T1
PF00258, G8BZP9
PF00258, A0A8H5GNH0
PF00258, A0A7S1X5T5
PF00258, A0A061SD83
PF00258, A0A7L3JP39
PF00258, K0T8C8
PF00258, B8LBR0
PF00258, A0A7J6VK40
PF00258, A0A8H7T040
PF00258, A0A4P9XIL7
PF00258, A0A6I9Y8B9
PF00258, A0A2R4LGJ7
PF00258, L8X3E8
PF00258, M5CHL2
PF00258, A0A140B164
PF00258, A0A140B161
PF00258, A0A0L0DWP0
PF00258, Q4UI24
PF00258, L1LFA5
PF00258, A0A976SKC7
PF00258, J4D9P2
PF00258, Q4N7I2
PF00258, A0A0N5D562
PF00258, A0A9P6HHY1
PF00258, A0A0C2J3M6
PF00258, A0A9P8W034
PF00258, A0AB32WUT4
PF00258, G2Q2J7
PF00258, A0A3S4BHA3
PF00258, G2R2L3
PF00258, A0A8D2JUC9
PF00258, A0A0F4ZBL8
PF00258, A0A7L1XTI0
PF00258, A0AAU9SZK6
PF00258, A0A1W0AB09
PF00258, A0A6P9A1X0
PF00258, A0A7K7Z4E9
PF00258, A0A507BBH3
PF00258, A0A850ZEW9
PF00258, A0A9W8DY33
PF00258, A0A151ZC52
PF00258, A0A553NV18
PF00258, C6JWF2
PF00258, A0A177UNN3
PF00258, A0A8X7MPV7
PF00258, A0AAN6K177
PF00258, A0A177TWC5
PF00258, A0A9N8LLW9
PF00258, A0A8X7T856
PF00258, A0A066V8R6
PF00258, A0A316Z8R0
PF00258, A0A7R9I757
PF00258, A0A7R9P858
PF00258, A0A7R9GRA1
PF00258, A0A7R8ZGT8
PF00258, A0A7R9JMR1
PF00258, A0A7R9HLF5
PF00258, A0A7R9DSS6
PF00258, A0A7R9FZG3
PF00258, A0A7R9NWJ8
PF00258, A0A7S1EQW7
PF00258, A0A099ZSN6
PF00258, A0A851DNI5
PF00258, A0A2K3QQF7
PF00258, A0A0L0NGD2
PF00258, A0A2S4KPK5
PF00258, A0AA49K9L2
PF00258, A0A1E4TB72
PF00258, G8ZVX8
PF00258, A0A7H9I129
PF00258, A0A9P4NZ74
PF00258, A0A3P7GNP1
PF00258, A0A7J6JV15
PF00258, S8G8K4
PF00258, S7V3V9
PF00258, V4ZQR4
PF00258, A0A139Y6N3
PF00258, A0A425HZ84
PF00258, A0A2G8YB35
PF00258, A0A086LC33
PF00258, A0A086KTM1
PF00258, A0A086QWH8
PF00258, A0A086M7P7
PF00258, A0A2T6IZL9
PF00258, A0A151HFK4
PF00258, A0A086QHE1
PF00258, A0A086L5D9
PF00258, A0A346QRP0
PF00258, A0A7K5J9M3
PF00258, L7JVC1
PF00258, A0A151JCF9
PF00258, A0A195F2A1
PF00258, A0A1Y2IA48
PF00258, A0AAD7X960
PF00258, A0A1M2VZN1
PF00258, Q8X1W0
PF00258, A0AAN7JKA5
PF00258, A0AAN7MEE8
PF00258, A0A2P5F6R3
PF00258, A0A6A6IYY9
PF00258, A0A4Q1BJN1
PF00258, A0A146KIF9
PF00258, A0AAN7A3N7
PF00258, A0AAN6XSH7
PF00258, A0A0V0G901
PF00258, A0A170W1G1
PF00258, D6X2B1
PF00258, A0A977XUW7
PF00258, A0A835ZFF6
PF00258, A0A2Y9R6J0
PF00258, A0A0V1DFY6
PF00258, A0A0V0UE04
PF00258, A0A1Y3EVV9
PF00258, A0A0V0SMI9
PF00258, A0A0V1N075
PF00258, A0A0V0ZGM2
PF00258, A0A0V1KFA0
PF00258, A0A0V1BS60
PF00258, A0A0V1I8M4
PF00258, A0AA85JT01
PF00258, A0AAN6ZCJ4
PF00258, A0A6G1HL16
PF00258, A0AAE1M362
PF00258, A0A395NQV2
PF00258, A0A6V8QMW3
PF00258, A0A2T3ZLE8
PF00258, A0A9W9EDF8
PF00258, A0A977IVX4
PF00258, A0A2T4BNB1
PF00258, A0A9P8QSG1
PF00258, A0A2P4ZDN9
PF00258, A0A1T3CNK2
PF00258, A0A2K0TK88
PF00258, A0A2T4AW38
PF00258, A0A9P4XH32
PF00258, A0A2T4CGD4
PF00258, A0A2H2ZFW2
PF00258, A0A9P8KYV2
PF00258, A0A8G0L3H6
PF00258, A0A9P8RLE0
PF00258, A0A6H5IKZ8
PF00258, A0A7G5CF69
PF00258, A0ABD2W573
PF00258, A0A852IW22
PF00258, A0A8H5GYB6
PF00258, A0A232FBN8
PF00258, A2FVK4
PF00258, A0A642V8E3
PF00258, A0A8X6KPF0
PF00258, A0A8X7C8C2
PF00258, F2Q595
PF00258, A0A9P5CWM9
PF00258, A0A059JJB9
PF00258, A0A178F7N5
PF00258, A0A080WS91
PF00258, A0A022VZH8
PF00258, A0A022XQQ7
PF00258, F2S0D1
PF00258, D4D6C0
PF00258, A0A178FDI6
PF00258, B3S4K3
PF00258, A0A7E5WYU8
PF00258, J6FAJ3
PF00258, K1WB62
PF00258, A0AAN8G6G5
PF00258, Q0PRK1
PF00258, A0A5S6QJD0
PF00258, A0A085NC61
PF00258, A0A077ZL47
PF00258, A0A7S2EB65
PF00258, A0A392PVV0
PF00258, A0A2K3PK57
PF00258, A0A2Z6NP50
PF00258, A0A9W7L9M0
PF00258, A0A9W7EJT3
PF00258, A0A9W7FBK9
PF00258, A0A9W7EC13
PF00258, A0A9W7ETN4
PF00258, A0A9W7ET09
PF00258, A0A9W7X485
PF00258, A0A5A9N887
PF00258, A0A7J7D1D4
PF00258, A0A9R1L5P2
PF00258, A0A9R1S3S0
PF00258, M7ZZB9
PF00258, A0A1J4KJR1
PF00258, A0A7L0ELB9
PF00258, A0A1V9XMW8
PF00258, A0A9P8UPF1
PF00258, Q7Z270
PF00258, Q583J8
PF00258, A0A3L6L896
PF00258, D0A7U1
PF00258, G0UUT7
PF00258, A0A422PJH0
PF00258, Q0MRD4
PF00258, Q4DS22
PF00258, V5DK85
PF00258, K2NRQ7
PF00258, A0A1G4IHH1
PF00258, A0A422NS70
PF00258, A0A061IZB1
PF00258, A0A1X0P9S6
PF00258, G0UC85
PF00258, A0A292Q9V6
PF00258, A0A2T7A1X9
PF00258, A0A317T0N8
PF00258, D5GL57
PF00258, A0A0C3MCS9
PF00258, A0A8H6M280
PF00258, L8YDG5
PF00258, Q9XSH2
PF00258, A0A9Q0J8C8
PF00258, A0A7L3LRC5
PF00258, A0A6J3R280
PF00258, A0A851R8W1
PF00258, A0A7L0XU60
PF00258, A0A093GNJ9
PF00258, A0AAD5EGX9
PF00258, A0A8H7PVH7
PF00258, A0ABD0XCI7
PF00258, C4RRN2
PF00258, A0A0B1P902
PF00258, A0A7K6BK89
PF00258, A0A7L3UK99
PF00258, A0ABC9C518
PF00258, A0A8D2H2S8
PF00258, A0A852L1G4
PF00258, A0A7K5T8G9
PF00258, A0A452QAI2
PF00258, A0A8M1G7R7
PF00258, A0A1B5L5G1
PF00258, A0A8H8QMM6
PF00258, I2G010
PF00258, A0A5C3EI51
PF00258, T0L975
PF00258, A0A0F9WUZ0
PF00258, C4V6W0
PF00258, A0AAX4JD78
PF00258, A7TG51
PF00258, A0A8B8IDD8
PF00258, A0A835VF49
PF00258, A0A7S4MI67
PF00258, A0A7D8YYF3
PF00258, A0AAF0Y444
PF00258, A0A8D2IVW8
PF00258, A0A7M7L435
PF00258, L2GSS0
PF00258, A0A517LBZ2
PF00258, A0A8H3YY06
PF00258, A0A4Z1PS25
PF00258, A0A370TX68
PF00258, A0AAV9QNW7
PF00258, A0A0D2ACA8
PF00258, C9STS6
PF00258, A0AA44W9F2
PF00258, G2WQ92
PF00258, A0A8I3ATG5
PF00258, A0A3M9YAY4
PF00258, A0A834NLG8
PF00258, A0ABD2D2F5
PF00258, A0A834P9T0
PF00258, A0ABD2BM20
PF00258, A0A834NGP3
PF00258, A0AAV0Z0B0
PF00258, Q43235
PF00258, A0A6I9I061
PF00258, A0A851L5C8
PF00258, A0A852EXG6
PF00258, A0A0S3RQD8
PF00258, A0AAQ3P229
PF00258, P37116
PF00258, A0A4D6MDJ3
PF00258, A0A7K5L718
PF00258, A0A6A6H8H3
PF00258, A0AA38YPB8
PF00258, F6H266
PF00258, A0A7S1KIL6
PF00258, A0A0G4EU79
PF00258, L2GN59
PF00258, A0A8J4BKZ0
PF00258, D8TSX8
PF00258, A0A8J4G546
PF00258, A0A4X2LRF7
PF00258, A0A3Q7RN22
PF00258, A0A4T0FMM5
PF00258, A0A4T0J8J0
PF00258, R9AMK4
PF00258, A0A4T0QA78
PF00258, I4YK22
PF00258, A0A6A6JBI3
PF00258, A0A2T0FN15
PF00258, A0A1E3P7Y6
PF00258, K0KHF4
PF00258, A0A9P8PZA2
PF00258, A0A9P8Q1G0
PF00258, D6P3J1
PF00258, A0A1C8EB25
PF00258, A0A2H3JZP7
PF00258, A0A0C9RLY5
PF00258, J9F9N6
PF00258, A0A6M2DRT0
PF00258, A0A974CPW8
PF00258, A0A8J0T1R8
PF00258, A0A3B5MXJ2
PF00258, M4AXQ7
PF00258, A0A7L3P849
PF00258, A0A9W8NGV3
PF00258, A0AAN7Z3M6
PF00258, A0A553I466
PF00258, A0A439DGE7
PF00258, A0A4Z0YX69
PF00258, A0A7C8ND97
PF00258, A0A165H8N2
PF00258, A0AAV1HA81
PF00258, A0A371C1J0
PF00258, Q6CFH1
PF00258, A0AAD5RZX1
PF00258, A0A6J2E5N7
PF00258, A0A1R1PYB7
PF00258, A0A7L3F9T2
PF00258, A0A6A6C925
PF00258, B8A0N6
PF00258, A0A0A1XJK9
PF00258, A0A8J5LY99
PF00258, A0A8J5VZ57
PF00258, A0A6P6G2Q4
PF00258, A0A978V853
PF00258, A0AAW1F2C6
PF00258, A0A8D2QKB6
PF00258, A0A067RNW1
PF00258, A0A7S2VPH0
PF00258, A0A6A6E0L0
PF00258, A0AA38HJT6
PF00258, A0A0K9PZ61
PF00258, A0A8K1GWK4
PF00258, A0A7L2K4S2
PF00258, A0A8D2QMW3
PF00258, A0A346M705
PF00258, A0A8J2T8C1
PF00258, A0A4C2EDE0
PF00258, B2G4E3
PF00258, C5E4A6
PF00258, A0A7H9AXN6
PF00258, A0A0F4GUK3
PF00258, F9XK00
PF00258, A0A1X7S2X2
PF00258, A0A1Y6LVL1
PF00258, A0A2H1GY27
PF00258, A0A1E4T551
PF00258, A0A9P0QRA7
PF00258, A0A8J5UUS3
PF00258, A0AAW1QRB0
PF00258, A0A0A1SSK4
"""

def get_fasta_sequences(text_data):
    # Extract IDs: split by line, then take the second part after the comma
    lines = [line.strip() for line in text_data.strip().split('\n') if ',' in line]
    uniprot_ids = [line.split(',')[1].strip() for line in lines]

    output_file = "sequences.fasta"
    base_url = "https://rest.uniprot.org/uniprotkb/"

    print(f"Starting retrieval of {len(uniprot_ids)} sequences...")

    with open(output_file, "w") as f:
        for i, uid in enumerate(uniprot_ids):
            try:
                # Fetch FASTA format from UniProt
                response = requests.get(f"{base_url}{uid}.fasta")
                if response.status_code == 200:
                    f.write(response.text + "\n")
                    print(f"[{i+1}/{len(uniprot_ids)}] Success: {uid}")
                else:
                    print(f"[{i+1}/{len(uniprot_ids)}] Failed: {uid} (Status {response.status_code})")

                # Small sleep to be polite to the server
                time.sleep(0.1)

            except Exception as e:
                print(f"Error fetching {uid}: {e}")

    print(f"\nDone! All sequences saved to {output_file}")

if __name__ == "__main__":
    get_fasta_sequences(raw_data)

Starting retrieval of 3730 sequences...
[1/3730] Success: A0A6A5XRE7
[2/3730] Success: A0ABD1V4T6
[3/3730] Success: A0A8B8JQF3
[4/3730] Success: A0A168RZW1
[5/3730] Success: A0A1X2IUX7
[6/3730] Success: A0AAE1KJ81
[7/3730] Success: L8HGF7
[8/3730] Success: A0A8B7Z8U5
[9/3730] Success: A0A091NBT3
[10/3730] Success: A0A498ST25
[11/3730] Success: A0A3Q1FIP1
[12/3730] Success: A0A9P0LPN2
[13/3730] Success: A0A812D275
[14/3730] Success: A0A316Z198
[15/3730] Success: A0A9N9JIZ9
[16/3730] Success: A0A8B9RPA5
[17/3730] Success: A0AAD5J8Q8
[18/3730] Success: A0AA39RQA5
[19/3730] Success: A0A5C7H9M4
[20/3730] Success: A0A6G1SKQ7
[21/3730] Success: A0AAN7C6Q3
[22/3730] Success: A0A1V9ZE56
[23/3730] Success: A0A6J2ANI1
[24/3730] Success: A0AAD8GD54
[25/3730] Success: A0A444UGC6
[26/3730] Success: A0AAF1BWT8
[27/3730] Success: A0AAV9FCX0
[28/3730] Success: A0AAV9BGZ6
[29/3730] Success: A0AAW2ZR25
[30/3730] Success: A0A455R5H4
[31/3730] Success: A0A914E098
[32/3730] Success: A0A7K7PTB0
[33/3730] Suc

This parses through each sheet and determines if a pathway is full in each domain

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

def analyze_strict_pathways_exact_names():
    # --- CONFIGURATION ---
    target_folder_name = "Lauren Paprocki"

    domains = {
        "Archaea": "Archaea_Pathway_Analysis_Outputs",
        "Bacteria": "Bacterial_Pathway_Analysis_Outputs",
        "Eukaryotes": "Eukaryotic_Pathway_Analysis_Outputs",
        "Viruses": "Viral_Pathway_Analysis_Outputs"
    }

    # 1. Find the Root Folder
    root_search = '/content/drive/My Drive'
    found_root = None
    for root, dirs, files in os.walk(root_search):
        if target_folder_name in dirs:
            found_root = os.path.join(root, target_folder_name)
            break

    if not found_root:
        print(f"❌ Error: Could not find '{target_folder_name}'.")
        return

    # --- STEP 2: PRE-SCAN FOR MAXIMUM STEPS ---
    # This identifies the "Full" requirement for every unique pathway name found
    required_step_counts = {}

    all_files = []
    for sub_name in domains.values():
        for root, dirs, files in os.walk(found_root):
            if sub_name in dirs:
                all_files.extend(glob.glob(os.path.join(root, sub_name, "*.xlsx")))

    print("Pre-scanning headers to establish strict step requirements...")
    for file_path in all_files:
        try:
            # We only read headers to save memory/time
            hdr = pd.read_excel(file_path, nrows=0).columns
            step_headers = [col for col in hdr if ' Step ' in col]
            for h in step_headers:
                # We split but keep the name EXACTLY as it appears
                base_name = h.split(' Step ')[0] # Preserves spaces/parentheses exactly
                step_num = int(h.split(' Step ')[1])
                if base_name not in required_step_counts or step_num > required_step_counts[base_name]:
                    required_step_counts[base_name] = step_num
        except:
            continue

    # --- STEP 3: PERFORM ANALYSIS ---
    master_results = {}

    for domain_label, sub_name in domains.items():
        target_sub_path = None
        for root, dirs, files in os.walk(found_root):
            if sub_name in dirs:
                target_sub_path = os.path.join(root, sub_name)
                break

        if not target_sub_path:
            continue

        excel_files = glob.glob(os.path.join(target_sub_path, "*.xlsx"))
        print(f"Analyzing {len(excel_files)} files in {domain_label}...")

        for file_path in excel_files:
            file_name = os.path.basename(file_path)
            pfam_id = file_name.split('_')[-1].replace('.xlsx', '')

            try:
                df = pd.read_excel(file_path, engine='openpyxl')
                if 'Status' in df.columns and df.shape[1] < 3:
                    continue

                step_headers = [col for col in df.columns if ' Step ' in col]
                # Identify every pathway base name in this specific file
                pathways_in_sheet = set([h.split(' Step ')[0] for h in step_headers])

                for path_name in pathways_in_sheet:
                    if path_name not in master_results:
                        # Initialize with exact name
                        master_results[path_name] = {d: {"Full": "No", "Source": "N/A"} for d in domains.keys()}

                    # Get all columns belonging to this exact pathway name
                    path_cols = [h for h in step_headers if h.startswith(path_name)]
                    actual_step_count_in_file = len(path_cols)
                    required_count = required_step_counts.get(path_name, 0)

                    # STRICT CHECK: File must have ALL steps, and row must have NO 'missing' values
                    if actual_step_count_in_file >= required_count and required_count > 0:
                        data_clean = df[path_cols].astype(str).apply(lambda x: x.str.lower().str.strip())
                        is_full_row = (data_clean != "missing").all(axis=1)

                        if is_full_row.any():
                            if master_results[path_name][domain_label]["Full"] == "No":
                                master_results[path_name][domain_label] = {
                                    "Full": "Yes",
                                    "Source": pfam_id
                                }
            except Exception as e:
                pass

    # --- STEP 4: COMPILE FINAL SUMMARY ---
    rows = []
    # We sort the exact names alphabetically for the final sheet
    for path_name in sorted(master_results.keys()):
        entry = {"Pathway Name": path_name}
        for d_label in domains.keys():
            entry[f"{d_label} Full"] = master_results[path_name][d_label]["Full"]
            entry[f"{d_label} Source Pfam"] = master_results[path_name][d_label]["Source"]
        rows.append(entry)

    if rows:
        final_df = pd.DataFrame(rows)
        # Ensuring names are not truncated or altered during export
        output_path = "/content/drive/My Drive/STRICT_EXACT_NAME_SUMMARY.xlsx"
        final_df.to_excel(output_path, index=False)
        print(f"\n🚀 SUCCESS! Strict summary with {len(final_df)} pathways created.")
        print(f"Saved to: {output_path}")
    else:
        print("❌ No valid pathway data found.")

analyze_strict_pathways_exact_names()

Mounted at /content/drive
Pre-scanning headers to establish strict step requirements...
Analyzing 81 files in Archaea...
Analyzing 81 files in Bacteria...
Analyzing 81 files in Eukaryotes...
Analyzing 81 files in Viruses...

🚀 SUCCESS! Strict summary with 29 pathways created.
Saved to: /content/drive/My Drive/STRICT_EXACT_NAME_SUMMARY.xlsx


there was some missing pathways so this code corrects for that

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_targeted_missing_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')

    # NEW FOLDER FOR MISSING DATA
    output_base = os.path.join(base_path, 'Missing_Pathway_Data')

    # Define the specific pathways to catch
    missing_targets = [
        "tRNA-uridine34 modifications (B. subtilis)",
        "tRNA-uridine34 modifications (E. coli)",
        "tRNA-uridine34 modifications (eukaryotic cytoplasm)",
        "tRNA-uridine34 modifications (mammalian mitochondria)",
        "tRNA wobble base lysidine biosynthesis",
        "tRNA dihydrouridine",
        "tRNA 5-methyldihydrouridine biosynthesis"
    ]

    domains = {
        "Viral": "Viruses",
        "Bacterial": "Bacteria",
        "Archaeal": "Archaea",
        "Eukaryotic": "Eukaryota"
    }

    if not os.path.exists(output_base):
        os.makedirs(output_base)

    # 2. Load Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)
        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Mappings
    gene_to_metadata = {str(row['Gene']).strip().upper(): str(row['pfam ID']).strip() for _, row in gene_mapping_df.iterrows()}

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        if p_name in missing_targets:
            steps = []
            gene_cols = [c for c in row.index if 'Gene' in c]
            for col in gene_cols:
                if pd.notna(row[col]) and str(row[col]).strip() != "":
                    step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                    steps.append(step_genes)
            path_to_steps[p_name] = steps

    # 4. File Indexing
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}
    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        if pfam_id in pfam_data_cache: return pfam_data_cache[pfam_id]
        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_map[str(row['Organism']).strip().lower()] = str(row['Entry']).strip()
            except: pass
        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Targeted Processing
    # Filter enzyme_df for ONLY the missing pathways
    filtered_enzymes = enzyme_df[enzyme_df['Pathway Name'].apply(lambda x: any(target in str(x) for target in missing_targets))]

    print(f"Processing {len(filtered_enzymes)} enzymes for targeted missing pathways...")

    for domain_prefix, taxon_keyword in domains.items():
        domain_folder = os.path.join(output_base, f"{domain_prefix}_Missing_Outputs")
        if not os.path.exists(domain_folder): os.makedirs(domain_folder)

        for idx, target_row in filtered_enzymes.iterrows():
            current_pfam = str(target_row['pfam ID']).strip()
            pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip() in missing_targets]

            if not pathway_list: continue

            primary_path = pfam_file_lookup.get(current_pfam)
            if not primary_path: continue

            # Load primary TSV and filter for domain
            primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
            lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
            mask = primary_df[lineage_col].str.contains(taxon_keyword, case=False, na=False)
            unique_orgs = sorted(primary_df[mask]['Organism'].unique())

            if not unique_orgs: continue

            results = []
            for org_name in unique_orgs:
                org_lower = org_name.strip().lower()
                row_data = {"Organism": org_name}
                for p_name in pathway_list:
                    steps = path_to_steps.get(p_name, [])
                    for i, step_genes in enumerate(steps, 1):
                        col_header = f"{p_name} Step {i}"
                        found = []
                        for gene in step_genes:
                            pid = gene_to_metadata.get(gene.upper())
                            if pid:
                                lookup = get_pfam_data(pid)
                                if org_lower in lookup: found.append(f"{pid}, {lookup[org_lower]}")
                        row_data[col_header] = "; ".join(list(set(found))) if found else "missing"
                results.append(row_data)

            if results:
                out_file = os.path.join(domain_folder, f'{domain_prefix}_Targeted_{current_pfam}.xlsx')
                pd.DataFrame(results).to_excel(out_file, index=False)

    print(f"\n✅ All missing pathway data has been generated in: {output_base}")

generate_targeted_missing_analysis()

Mounted at /content/drive
Connecting to Google Sheets...
Processing 24 enzymes for targeted missing pathways...

✅ All missing pathway data has been generated in: /content/drive/My Drive/Lauren Paprocki/Missing_Pathway_Data
